# Sketch2Photo — master training and fair comparison

**Use this notebook for the new experiment.** It replaces the earlier 23-cell
diffusion-only proposal. Training is in Colab; no files from VS Code are required.

Both tasks are trained separately: **photo → pencil** and **pencil → RGB photograph**.
For EACH task, compare three genuinely different approaches on identical splits:

| Approach | Architecture / training | Starting weights |
|---|---|---|
| A | Skip-connected U-Net, L1 + perceptual + gradient losses | Random |
| B | Nine-block residual generator + conditional PatchGAN, paired and adversarial losses | Random |
| C | Eight-channel InstructPix2Pix latent diffusion, LoRA fine-tuning | Cited pretrained image editor |

This means **six trained models**, followed by LR/weight-decay trials on the
validation-winning approach in each direction. One notebook does not mean one model.
The app's default for each direction is chosen using validation, not test results.

## Honest quality contract

No new production model has been trained here. Code checks are not quality evidence.
Neither more cells, more epochs, nor more compute guarantees an improvement. This
notebook supplies measurable selection and failure analysis instead of promising
perfect drawings, perfect faces, or exact original colours for every sketch.
Sketches discard colour information; plausible colour is possible, unique recovery is not.
Pretrained models may hallucinate identity/details and may have seen benchmark images
before pretraining; their data/compute advantage must be disclosed in the comparison.

512px is the native working resolution for this pretrained editor, not a promise of
high-resolution artist detail. Do not simply change it to 1024 and call it higher quality.
The full run is quality-oriented and may take much longer than the assignment's
suggested free-Colab budget. Confirm that expanded scope with your teacher.

## Final quality-focused revision

+The default is now the complete run rather than the tiny pilot. The official FS2K
+download is enabled, real artist pairs are deliberately sampled during training,
+white-paper sketch inputs receive mild deterministic style augmentation, and the
+paired CNN/GAN loss explicitly protects colour chrominance in sketch-to-photo.
+Before expensive training begins, the notebook pauses after displaying actual
+photo/sketch pairs and requires the user to type `TRAIN`. This prevents an unnoticed
+bad dataset path or drawing style from wasting a long run.
+
+**Keep old checkpoints.** They are archived baselines, not deleted or overwritten.
+Their original 128px results use different splits and cannot be copied into the new
fair-comparison table. A new model earns replacement only after evaluation.


> **Corrected v2 workflow:** the notebook defaults to a fail-closed pilot. It uses a
> structure-preserving residual U-Net, meaningful pilot budgets, stronger chroma loss,
> higher diffusion image conditioning, and manual visual approval before test/export.
> Pencil-to-photo colours remain plausible estimates; exact hidden original colours cannot
> be recovered from a monochrome sketch alone.


> **Focused refinement phase:** the completed pilot selected U-Net in both directions.
> This notebook now warm-starts only those checkpoints, runs the required LR/regularization
> search, and performs 8,192 additional updates per direction. It does not repeat failed GAN
> or diffusion training.


> **Face-specific rescue:** Section 7 showed that mixed COCO/FS2K training improved
> LPIPS but still produced washed-out pencil and sepia face reconstruction. This phase
> uses all real aligned FS2K face pairs, removes the source-gradient constraint that
> preserved photographic shading in photo→pencil, and uses a direct RGB head with
> explicit colour statistics for sketch→photo. It does not repeat GAN or diffusion.


> **Final sketch→photo sharpening gate:** the all-FS2K direct U-Net restored plausible
> colour but remained visibly averaged. Photo→pencil is frozen. This short phase fine-tunes
> only sketch→photo with a conditional PatchGAN texture signal while retaining paired,
> perceptual, edge, structure and colour supervision. Intermediate snapshots are saved.


> **Dual-direction final refinement:** both face directions now receive a short conditional
> PatchGAN refinement. Photo→pencil uses stronger target-edge, Laplacian and neutral-graphite
> supervision; sketch→photo uses direct RGB, structure, chroma and colour-statistic supervision.
> Both start from the completed face-specific checkpoints and retain them as fallbacks.


In [ ]:
# Install in a fresh Colab runtime. Leave Colab's matching torch/torchvision pair in place.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.55.4', 'peft==0.17.1',
    'accelerate==1.10.1', 'huggingface-hub==0.34.4', 'safetensors==0.6.2',
    'opencv-python-headless==4.11.0.86', 'Pillow>=10', 'matplotlib',
    'requests', 'tqdm', 'scikit-image==0.25.2', 'streamlit==1.65.0', 'lpips==0.1.4', 'gdown==5.2.0'])
print('If Colab asks for a restart, restart the session once, then continue below.')


In [ ]:
from pathlib import Path
import os, sys, json, csv, math, random, hashlib, time, gc, shutil, zipfile, io
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from diffusers import StableDiffusionInstructPix2PixPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

assert torch.cuda.is_available(), 'Choose a Colab GPU runtime before training.'
DEVICE = 'cuda'
DTYPE = torch.float16
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('Local disk free:', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')

import torchvision, subprocess, platform
from collections import Counter, defaultdict
from contextlib import contextmanager
from torch import nn
import lpips
print('torchvision:', torchvision.__version__)
# Keep Colab's matching torch/torchvision wheels. The exact environment is saved below.


## 1. Stage A - face-specific refinement

This completed stage improves portrait translation using all audited FS2K pairs. It is
retained for reproducibility and as a fallback. Stage B below uses a persistent,
deterministic 20,000-image COCO subset to restore general-scene coverage and refines
only the weak pencil-to-photo direction.


In [ ]:
MODE = 'dual_adversarial_rescue'
DATA_SOURCE = 'fs2k_faces'
EXPERIMENT = 'dual_direction_adversarial_rescue_v1'
USE_GOOGLE_DRIVE = True
DOWNLOAD_FS2K = True

PREVIOUS_EXPERIMENT = 'face_specific_rescue_v1'
UPDATE_BUDGETS = {'photo_to_pencil': 4096, 'sketch_to_photo': 4096}
OUTPUT_MODES = {'photo_to_pencil': 'residual', 'sketch_to_photo': 'direct'}
FINAL_MODELS_APPROVED = False
DEPLOY_APPROACH = {'photo_to_pencil': None, 'sketch_to_photo': None}

CFG = {
    'resolution':512,'seed':SEED,'cnn_width':64,
    'generator_lr_by_direction':{'photo_to_pencil':5e-5,'sketch_to_photo':5e-5},
    'discriminator_lr_by_direction':{'photo_to_pencil':1e-4,'sketch_to_photo':1e-4},
    'cnn_weight_decay':1e-4,'cnn_validate_every':512,'cnn_save_every':512,
    'metric_images_per_domain':16,
    'photo_aug_probability':0.30,'sketch_aug_probability':0.45,
    'pencil_strength':0.95,'pencil_background_fade':0.25,'pencil_texture':0.30,
    'l1_weight_by_direction':{'photo_to_pencil':6.0,'sketch_to_photo':6.0},
    'perceptual_weight_by_direction':{'photo_to_pencil':2.0,'sketch_to_photo':3.0},
    'gradient_weight_by_direction':{'photo_to_pencil':5.0,'sketch_to_photo':3.0},
    'laplacian_weight_by_direction':{'photo_to_pencil':2.0,'sketch_to_photo':1.5},
    'adversarial_weight_by_direction':{'photo_to_pencil':0.10,'sketch_to_photo':0.15},
    'pencil_gray_weight':3.0,'structure_weight':0.75,
    'chroma_weight':8.0,'color_stats_weight':3.0,'inference_steps':30,
}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_ROOT=Path('/content/drive/MyDrive/Sketch2Photo_training')
else:
    SAVE_ROOT=Path('/content/Sketch2Photo_training')
CACHE=Path('/content/sketch2photo_data')
PREVIOUS_ROOT=SAVE_ROOT/PREVIOUS_EXPERIMENT/'face_rescue'
RUN_ROOT=SAVE_ROOT/EXPERIMENT/MODE
STUDY_ROOT=RUN_ROOT
CACHE.mkdir(parents=True,exist_ok=True); RUN_ROOT.mkdir(parents=True,exist_ok=True)
assert (PREVIOUS_ROOT/'selected_models.json').is_file(),(
    'Missing completed face-specific run: '+str(PREVIOUS_ROOT/'selected_models.json'))
FS2K_ROOT='/content/FS2K'
STUDENT_NAME='Srun Chankhemara'
atomic_environment={'python':sys.version,'torch':torch.__version__,'torchvision':torchvision.__version__,
    'gpu':torch.cuda.get_device_name(0),'platform':platform.platform(),
    'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30}
(STUDY_ROOT/'environment.json').write_text(json.dumps(atomic_environment,indent=2))
(STUDY_ROOT/'requirements-lock.txt').write_text(
    subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Starting from completed face-specific checkpoints:',PREVIOUS_ROOT)
print('Saving dual-direction refinement to:',RUN_ROOT)
print('Each direction runs 4096 resumable updates; previous checkpoints remain untouched.')


IMPLEMENTATION_SHA = '911776b154fb5c248c38ca72141f20fa807afdb4d3fb958f369a22c3b7061c76'


In [ ]:
MODULE_SOURCES = {'src/__init__.py': '"""Local Sketch2Photo application and checkpoint-compatible model definitions."""\n',
 'src/approaches/__init__.py': '"""Architectures for the new, paired 512px comparison (not legacy '
                               'weights)."""\n'
                               '\n'
                               'from .unet import PencilUNet\n'
                               'from .gan import ResnetGenerator, PatchDiscriminator\n'
                               '\n'
                               '\n'
                               'def make_generator(approach, width=64, output_mode="residual"):\n'
                               "    if approach == 'unet':\n"
                               '        return PencilUNet(width, output_mode=output_mode)\n'
                               "    if approach == 'gan':\n"
                               '        return ResnetGenerator(width)\n'
                               '    raise ValueError(approach)\n',
 'src/approaches/diffusion.py': '"""Approach C: LoRA adaptation and InstructPix2Pix training objective.\n'
                                '\n'
                                'The notebook owns shared data, validation and resumable training '
                                'orchestration.\n'
                                'Target latents are VAE-scaled; source-image conditioning latents MUST '
                                'remain unscaled.\n'
                                '"""\n'
                                'import torch\n'
                                '\n'
                                '\n'
                                'def configure_adapter(pipe, rank):\n'
                                '    from peft import LoraConfig, get_peft_model\n'
                                '    assert pipe.unet.config.in_channels == 8\n'
                                '    pipe.vae.requires_grad_(False).eval().to(dtype=torch.float32)\n'
                                '    pipe.text_encoder.requires_grad_(False).eval()\n'
                                '    pipe.unet.requires_grad_(False)\n'
                                '    config = LoraConfig(r=rank, lora_alpha=rank, '
                                "init_lora_weights='gaussian',\n"
                                "                       target_modules=['to_q', 'to_k', 'to_v', 'to_out.0', "
                                "'conv_in'])\n"
                                '    pipe.unet = get_peft_model(pipe.unet, config)\n'
                                '    pipe.unet.enable_gradient_checkpointing()\n'
                                '    for param in pipe.unet.parameters():\n'
                                '        if param.requires_grad:\n'
                                '            param.data = param.data.float()\n'
                                '    return pipe\n'
                                '\n'
                                '\n'
                                'def noise_prediction_loss(pipe, scheduler, embeddings, batch, *, device, '
                                'dtype,\n'
                                '                          conditioning_dropout=.05, snr_gamma=5., '
                                'training=True,\n'
                                '                          fixed_seed=None, timestep=None):\n'
                                '    generator = None if fixed_seed is None else '
                                'torch.Generator(device=device).manual_seed(fixed_seed)\n'
                                "    source = batch['source'].to(device, dtype=torch.float32)\n"
                                "    target = batch['target'].to(device, dtype=torch.float32)\n"
                                '    with torch.no_grad():\n'
                                '        posterior = pipe.vae.encode(target).latent_dist\n'
                                '        clean = posterior.sample(generator=generator) if training else '
                                'posterior.mode()\n'
                                '        clean = (clean * pipe.vae.config.scaling_factor).to(dtype)\n'
                                '        source_latents = '
                                'pipe.vae.encode(source).latent_dist.mode().to(dtype)\n'
                                '    size = clean.shape[0]\n'
                                '    noise = torch.randn(clean.shape, generator=generator, device=device, '
                                'dtype=dtype)\n'
                                '    timesteps = (torch.randint(scheduler.config.num_train_timesteps, '
                                '(size,), device=device, generator=generator)\n'
                                '                 if timestep is None else torch.full((size,), timestep, '
                                'device=device, dtype=torch.long))\n'
                                '    noisy = scheduler.add_noise(clean, noise, timesteps)\n'
                                '    text = embeddings[:1].expand(size, -1, -1)\n'
                                '    if training:\n'
                                '        p = conditioning_dropout\n'
                                '        draw = torch.rand(size, device=device)\n'
                                '        text = torch.where((draw < 2*p).view(size, 1, 1), '
                                'embeddings[1:].expand_as(text), text)\n'
                                '        source_latents = source_latents * (~((draw >= p) & (draw < '
                                '3*p))).view(size, 1, 1, 1)\n'
                                "    with torch.autocast(device_type=str(device).split(':')[0], dtype=dtype, "
                                'enabled=dtype != torch.float32):\n'
                                '        prediction = pipe.unet(torch.cat([noisy, source_latents], dim=1), '
                                'timesteps,\n'
                                '                               encoder_hidden_states=text).sample\n'
                                '    prediction_type = scheduler.config.prediction_type\n'
                                "    if prediction_type == 'epsilon':\n"
                                '        objective = noise\n'
                                "    elif prediction_type == 'v_prediction':\n"
                                '        objective = scheduler.get_velocity(clean, noise, timesteps)\n'
                                '    else:\n'
                                '        raise ValueError(prediction_type)\n'
                                '    loss = (prediction.float()-objective.float()).square().mean(dim=(1, 2, '
                                '3))\n'
                                '    alpha = scheduler.alphas_cumprod.to(device)[timesteps]\n'
                                '    snr = alpha/(1-alpha).clamp_min(1e-8)\n'
                                '    weights = snr.clamp(max=snr_gamma)/(snr+(prediction_type == '
                                "'v_prediction')).clamp_min(1e-8)\n"
                                '    return (loss*weights).mean()\n',
 'src/approaches/gan.py': '"""Approach B: residual generator plus conditional PatchGAN; no U-Net skips.\n'
                          '\n'
                          'Uses adversarial, paired reconstruction and perceptual losses in the notebook.\n'
                          'Inspired by the pix2pix/CycleGAN project; this is a separate PyTorch '
                          'implementation.\n'
                          '"""\n'
                          'from torch import nn\n'
                          '\n'
                          '\n'
                          'class Residual(nn.Module):\n'
                          '    def __init__(self, channels):\n'
                          '        super().__init__()\n'
                          '        self.layers = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(channels, '
                          'channels, 3),\n'
                          '            nn.InstanceNorm2d(channels, affine=True), nn.ReLU(), '
                          'nn.ReflectionPad2d(1),\n'
                          '            nn.Conv2d(channels, channels, 3), nn.InstanceNorm2d(channels, '
                          'affine=True))\n'
                          '\n'
                          '    def forward(self, x):\n'
                          '        return x + self.layers(x)\n'
                          '\n'
                          '\n'
                          'class ResnetGenerator(nn.Module):\n'
                          '    def __init__(self, width=64):\n'
                          '        super().__init__()\n'
                          '        layers = [nn.ReflectionPad2d(3), nn.Conv2d(3, width, 7),\n'
                          '                  nn.InstanceNorm2d(width, affine=True), nn.ReLU()]\n'
                          '        for a, b in ((width, width*2), (width*2, width*4)):\n'
                          '            layers += [nn.Conv2d(a, b, 3, stride=2, padding=1),\n'
                          '                       nn.InstanceNorm2d(b, affine=True), nn.ReLU()]\n'
                          '        layers += [Residual(width*4) for _ in range(9)]\n'
                          '        for a, b in ((width*4, width*2), (width*2, width)):\n'
                          "            layers += [nn.Upsample(scale_factor=2, mode='bilinear', "
                          'align_corners=False),\n'
                          '                       nn.Conv2d(a, b, 3, padding=1), nn.InstanceNorm2d(b, '
                          'affine=True), nn.ReLU()]\n'
                          '        layers += [nn.ReflectionPad2d(3), nn.Conv2d(width, 3, 7), nn.Tanh()]\n'
                          '        self.layers = nn.Sequential(*layers)\n'
                          '\n'
                          '    def forward(self, x):\n'
                          '        return self.layers(x)\n'
                          '\n'
                          '\n'
                          'class PatchDiscriminator(nn.Module):\n'
                          '    def __init__(self, width=64):\n'
                          '        super().__init__()\n'
                          '        layers = [nn.utils.spectral_norm(nn.Conv2d(6, width, 4, 2, 1)), '
                          'nn.LeakyReLU(.2)]\n'
                          '        for a, b, stride in ((width, width*2, 2), (width*2, width*4, 2), '
                          '(width*4, width*8, 1)):\n'
                          '            layers += [nn.utils.spectral_norm(nn.Conv2d(a, b, 4, stride, 1)), '
                          'nn.LeakyReLU(.2)]\n'
                          '        layers += [nn.utils.spectral_norm(nn.Conv2d(width*8, 1, 4, 1, 1))]\n'
                          '        self.layers = nn.Sequential(*layers)\n'
                          '\n'
                          '    def forward(self, paired_images):\n'
                          '        return self.layers(paired_images)\n',
 'src/approaches/unet.py': '"""Approach A: skip-connected U-Net trained with paired reconstruction '
                           'losses."""\n'
                           'import torch\n'
                           'from torch import nn\n'
                           'from torch.nn import functional as F\n'
                           '\n'
                           '\n'
                           'def block(a, b):\n'
                           '    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.GroupNorm(8, b), '
                           'nn.SiLU(),\n'
                           '                         nn.Conv2d(b, b, 3, padding=1), nn.GroupNorm(8, b), '
                           'nn.SiLU())\n'
                           '\n'
                           '\n'
                           'class PencilUNet(nn.Module):\n'
                           '    def __init__(self, width=64, output_mode="residual"):\n'
                           '        super().__init__()\n'
                           '        if output_mode not in {"residual", "direct"}:\n'
                           '            raise ValueError(f"Unsupported output mode: {output_mode}")\n'
                           '        self.output_mode = output_mode\n'
                           '        self.enc = nn.ModuleList([block(3, width), block(width, width*2),\n'
                           '                                 block(width*2, width*4), block(width*4, '
                           'width*8)])\n'
                           '        self.mid = block(width*8, width*8)\n'
                           '        self.dec = nn.ModuleList([block(width*16, width*4), block(width*8, '
                           'width*2),\n'
                           '                                 block(width*4, width), block(width*2, width)])\n'
                           '        self.out = nn.Conv2d(width, 3, 1)\n'
                           '        if output_mode == "residual":\n'
                           '            # Identity initialization is useful when the input and output have '
                           'similar\n'
                           '            # appearance (photo -> pencil).  It is deliberately not used for '
                           'the\n'
                           '            # direct RGB reconstruction head because that otherwise biases a '
                           'colour\n'
                           '            # model toward the monochrome sketch input.\n'
                           '            nn.init.zeros_(self.out.weight)\n'
                           '            nn.init.zeros_(self.out.bias)\n'
                           '\n'
                           '    def forward(self, x):\n'
                           '        source = x\n'
                           '        skips = []\n'
                           '        for layer in self.enc:\n'
                           '            x = layer(x); skips.append(x); x = F.avg_pool2d(x, 2)\n'
                           '        x = self.mid(x)\n'
                           '        for layer, skip in zip(self.dec, reversed(skips)):\n'
                           "            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', "
                           'align_corners=False)\n'
                           '            x = layer(torch.cat([x, skip], dim=1))\n'
                           '        prediction = self.out(x)\n'
                           '        if self.output_mode == "residual":\n'
                           '            return (source + prediction).clamp(-1, 1)\n'
                           '        return torch.tanh(prediction)\n',
 'src/diffusion_backend.py': '"""Inference for the separate Colab dual-direction diffusion export.\n'
                             '\n'
                             'Dependencies are loaded only when a model is opened. Legacy CNN inference is '
                             'separate.\n'
                             'The same source is embedded in the standalone Colab notebook and its exported '
                             'bundle.\n'
                             '"""\n'
                             '\n'
                             'import json\n'
                             'from pathlib import Path\n'
                             '\n'
                             "BASE_MODEL = 'timbrooks/instruct-pix2pix'\n"
                             "BASE_REVISION = '31519b5cb02a7fd89b906d88731cd4d6a7bbf88d'\n"
                             'PROMPTS = {\n'
                             "    'photo_to_pencil': (\n"
                             "        'Turn this image into a detailed hand-drawn graphite pencil drawing on "
                             "clean white paper. '\n"
                             "        'Use fine natural pencil strokes, delicate cross-hatching, gentle "
                             "tonal shading and clean highlights; '\n"
                             "        'avoid ink outlines, charcoal blocks, colour and digital-painting "
                             "texture. Preserve identity, anatomy, '\n"
                             "        'pose, proportions and composition.'\n"
                             '    ),\n'
                             "    'sketch_to_photo': (\n"
                             "        'Turn this pencil drawing into a sharp realistic full-colour "
                             "photograph with natural skin tones, '\n"
                             "        'distinct believable object colours, realistic lighting and detailed "
                             "photographic textures; avoid '\n"
                             "        'monochrome, sepia, illustration, plastic skin and blur. Preserve "
                             "identity, anatomy, pose, proportions '\n"
                             "        'and composition.'\n"
                             '    ),\n'
                             '}\n'
                             '\n'
                             '\n'
                             'def rgb_on_white(image):\n'
                             '    from PIL import Image, ImageOps\n'
                             '    image = ImageOps.exif_transpose(image)\n'
                             "    rgba = image.convert('RGBA')\n"
                             "    white = Image.new('RGBA', rgba.size, 'white')\n"
                             "    return Image.alpha_composite(white, rgba).convert('RGB')\n"
                             '\n'
                             '\n'
                             'def fit_canvas(image, size=512):\n'
                             '    """Aspect-preserving resize and padding, also used for training pairs."""\n'
                             '    from PIL import Image\n'
                             '    image = rgb_on_white(image)\n'
                             '    scale = size / max(image.size)\n'
                             '    width, height = [max(1, min(size, round(v * scale))) for v in image.size]\n'
                             '    resized = image.resize((width, height), Image.Resampling.LANCZOS)\n'
                             '    left, top = (size - width) // 2, (size - height) // 2\n'
                             "    canvas = Image.new('RGB', (size, size), 'white')\n"
                             '    canvas.paste(resized, (left, top))\n'
                             '    return canvas, (left, top, left + width, top + height)\n'
                             '\n'
                             '\n'
                             'def open_pipeline(adapter_dir=None, device=None, base_model=BASE_MODEL,\n'
                             '                  revision=BASE_REVISION, offload=False):\n'
                             '    import torch\n'
                             '    from diffusers import StableDiffusionInstructPix2PixPipeline, '
                             'DDIMScheduler\n'
                             '    from peft import PeftModel\n'
                             '    if device is None:\n'
                             "        device = 'cuda' if torch.cuda.is_available() else (\n"
                             "            'mps' if torch.backends.mps.is_available() else 'cpu')\n"
                             "    dtype = torch.float16 if device == 'cuda' else torch.float32\n"
                             '    pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(\n'
                             '        base_model, revision=revision, torch_dtype=dtype, '
                             'use_safetensors=True,\n'
                             "        variant='fp16' if dtype == torch.float16 else None)\n"
                             '    if adapter_dir is not None:\n'
                             '        adapter_dir = Path(adapter_dir)\n'
                             "        if not (adapter_dir / 'adapter_model.safetensors').is_file():\n"
                             "            raise FileNotFoundError(f'No trained adapter at {adapter_dir}')\n"
                             '        pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_dir), '
                             'is_trainable=False)\n'
                             '        pipe.unet.to(dtype=dtype)\n'
                             '    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)\n'
                             '    pipe.enable_vae_slicing()\n'
                             '    pipe.set_progress_bar_config(disable=True)\n'
                             "    if offload and device == 'cuda':\n"
                             '        pipe.enable_model_cpu_offload()\n'
                             '    else:\n'
                             '        pipe.to(device)\n'
                             '    return pipe\n'
                             '\n'
                             '\n'
                             'def generate(pipe, image, direction, seed=42, steps=30, text_guidance=6.5,\n'
                             "             image_guidance=2.2, size=512, description=''):\n"
                             '    import torch\n'
                             '    if direction not in PROMPTS:\n'
                             "        raise ValueError(f'Unknown direction: {direction}')\n"
                             "    if direction == 'sketch_to_photo':\n"
                             "        image = rgb_on_white(image).convert('L').convert('RGB')\n"
                             '    canvas, box = fit_canvas(image, size)\n'
                             '    # CPU generator works across CPU, CUDA, and MPS and is repeatable on each '
                             'backend.\n'
                             "    generator = torch.Generator(device='cpu').manual_seed(int(seed))\n"
                             '    prompt = PROMPTS[direction]\n'
                             '    if description.strip():\n'
                             "        prompt += ' ' + description.strip()\n"
                             '    with torch.inference_mode():\n'
                             '        output = pipe(prompt=prompt, image=canvas, '
                             'num_inference_steps=int(steps),\n'
                             '                      guidance_scale=float(text_guidance),\n'
                             '                      image_guidance_scale=float(image_guidance), '
                             'generator=generator)\n'
                             '    if output.nsfw_content_detected is not None and '
                             'any(output.nsfw_content_detected):\n'
                             "        raise RuntimeError('The base model filtered this output; it is not a "
                             "successful prediction.')\n"
                             '    result = output.images[0].crop(box)\n'
                             "    if direction == 'photo_to_pencil':\n"
                             "        result = result.convert('L').convert('RGB')\n"
                             '    return result\n'
                             '\n'
                             '\n'
                             'class DualEditor:\n'
                             '    """Load one direction at a time to limit GPU memory."""\n'
                             '    def __init__(self, bundle_dir, device=None):\n'
                             '        self.root = Path(bundle_dir)\n'
                             "        self.manifest = json.loads((self.root / 'manifest.json').read_text())\n"
                             "        if self.manifest.get('format') != 'sketch2photo-ip2p-lora-v1':\n"
                             "            raise ValueError('Expected a dual-direction notebook export, not a "
                             "legacy .pt checkpoint.')\n"
                             '        self.device = device\n'
                             '        self.pipe = None\n'
                             '        self.direction = None\n'
                             '\n'
                             "    def __call__(self, image, direction, seed=42, description=''):\n"
                             '        import gc\n'
                             '        import torch\n'
                             '        if image is None:\n'
                             "            raise ValueError('Upload an image first.')\n"
                             '        if direction not in PROMPTS:\n'
                             "            raise ValueError('Choose a supported direction.')\n"
                             '        if self.direction != direction:\n'
                             '            self.pipe = None\n'
                             '            gc.collect()\n'
                             '            if torch.cuda.is_available():\n'
                             '                torch.cuda.empty_cache()\n'
                             "            if hasattr(torch, 'mps') and torch.backends.mps.is_available():\n"
                             '                torch.mps.empty_cache()\n'
                             '            self.pipe = open_pipeline(self.root / direction, self.device,\n'
                             "                                      self.manifest['base_model'], "
                             "self.manifest['base_revision'],\n"
                             '                                      offload=True)\n'
                             '            self.direction = direction\n'
                             "        settings = self.manifest['inference']\n"
                             '        return generate(self.pipe, image, direction, seed=seed, '
                             'description=description, **settings)\n'
                             '\n'
                             '\n'
                             'def launch_demo(bundle_dir, port=8094):\n'
                             '    import gradio as gr\n'
                             '    editor = DualEditor(bundle_dir)\n'
                             "    with gr.Blocks(title='Sketch2Photo — trained diffusion adapters') as "
                             'demo:\n'
                             "        gr.Markdown('# Sketch2Photo\\nGenerate a pencil drawing or a colour "
                             "photograph.')\n"
                             "        direction = gr.Radio(list(PROMPTS), value='photo_to_pencil', "
                             "label='Direction')\n"
                             '        with gr.Row():\n'
                             "            source = gr.Image(type='pil', label='Your image')\n"
                             "            result = gr.Image(type='pil', label='Generated result', "
                             "interactive=False, format='png')\n"
                             "        description = gr.Textbox(label='Optional description or desired "
                             "colours')\n"
                             "        seed = gr.Number(value=42, precision=0, label='Seed')\n"
                             "        gr.Button('Generate', variant='primary').click(\n"
                             '            editor, [source, direction, seed, description], result, '
                             'concurrency_limit=1)\n'
                             "        gr.Markdown('Colours are plausible estimates. Fine facial details can "
                             "change. '\n"
                             "                    'Results are generated from the uploaded image using your "
                             "selected trained adapter.')\n"
                             '    demo.queue(default_concurrency_limit=1).launch(\n'
                             "        server_name='127.0.0.1', server_port=int(port), show_error=True)\n"
                             '\n'
                             '\n'
                             "if __name__ == '__main__':\n"
                             '    import argparse\n'
                             '    parser = argparse.ArgumentParser()\n'
                             "    parser.add_argument('--bundle', required=True)\n"
                             "    parser.add_argument('--port', type=int, default=8094)\n"
                             '    args = parser.parse_args()\n'
                             '    launch_demo(args.bundle, args.port)\n',
 'src/study_backend.py': '"""Shared inference for the three-approach Colab study export."""\n'
                         'import gc\n'
                         'import json\n'
                         'from pathlib import Path\n'
                         'from .diffusion_backend import PROMPTS, fit_canvas, open_pipeline, generate\n'
                         '\n'
                         '\n'
                         'class StudyEditor:\n'
                         '    def __init__(self, bundle_dir, device=None):\n'
                         '        import torch\n'
                         '        self.root = Path(bundle_dir)\n'
                         "        self.manifest = json.loads((self.root / 'manifest.json').read_text())\n"
                         "        if self.manifest.get('format') != 'sketch2photo-study-v2':\n"
                         "            raise ValueError('Expected the master notebook study export.')\n"
                         "        self.device = device or ('cuda' if torch.cuda.is_available() else (\n"
                         "            'mps' if hasattr(torch.backends, 'mps') and "
                         "torch.backends.mps.is_available() else 'cpu'))\n"
                         '        self.model, self.key = None, None\n'
                         '\n'
                         "    def __call__(self, image, direction, approach='validation_best', seed=42, "
                         "description=''):\n"
                         '        import numpy as np\n'
                         '        import torch\n'
                         '        from PIL import Image\n'
                         '        from .approaches import make_generator\n'
                         '        if image is None:\n'
                         "            raise ValueError('Upload a photo or pencil sketch first.')\n"
                         '        if direction not in PROMPTS:\n'
                         "            raise ValueError('Unknown direction')\n"
                         "        if approach == 'validation_best':\n"
                         "            approach = self.manifest['validation_best'][direction]\n"
                         '        key = (direction, approach)\n'
                         "        info = self.manifest['models'][direction][approach]\n"
                         '        if key != self.key:\n'
                         '            self.model = None; self.key = None; gc.collect()\n'
                         '            if torch.cuda.is_available():\n'
                         '                torch.cuda.empty_cache()\n'
                         "            if hasattr(torch.backends, 'mps') and "
                         'torch.backends.mps.is_available():\n'
                         '                torch.mps.empty_cache()\n'
                         "            if approach == 'diffusion':\n"
                         "                self.model = open_pipeline(self.root / info['path'], self.device,\n"
                         "                    self.manifest['base_model'], self.manifest['base_revision'], "
                         'offload=True)\n'
                         '            else:\n'
                         "                state = torch.load(self.root / info['path'], map_location='cpu', "
                         'weights_only=True)\n'
                         "                assert state['approach'] == approach and state['direction'] == "
                         'direction\n'
                         "                self.model = make_generator(approach, state['width'])\n"
                         "                self.model.load_state_dict(state['generator'], strict=True)\n"
                         '                self.model.to(self.device).eval()\n'
                         '            self.key = key\n'
                         "        settings = self.manifest['inference']\n"
                         "        if approach == 'diffusion':\n"
                         '            return generate(self.model, image, direction, seed=seed, '
                         'description=description, **settings)\n'
                         "        canvas, box = fit_canvas(image, settings['size'])\n"
                         "        if direction == 'sketch_to_photo':\n"
                         "            canvas = canvas.convert('L').convert('RGB')\n"
                         '        x = torch.from_numpy(np.asarray(canvas, '
                         'dtype=np.float32).copy().transpose(2, 0, 1) / 127.5 - 1)\n'
                         '        with torch.inference_mode():\n'
                         '            y = self.model(x[None].to(self.device))[0].float().cpu().clamp(-1, 1)\n'
                         '        image = Image.fromarray(((y.permute(1, 2, '
                         "0).numpy()+1)*127.5).round().astype('uint8')).crop(box)\n"
                         "        return image.convert('L').convert('RGB') if direction == 'photo_to_pencil' "
                         'else image\n'
                         '\n'
                         '\n'
                         'def launch_demo(bundle_dir, port=8088):\n'
                         '    import gradio as gr\n'
                         '    editor = StudyEditor(bundle_dir)\n'
                         "    with gr.Blocks(title='Sketch2Photo — trained model comparison') as demo:\n"
                         "        gr.Markdown('# Sketch2Photo\\nBoth directions use trained models from your "
                         "Colab export. '\n"
                         "                    'Original colours cannot be uniquely recovered from a "
                         "sketch.')\n"
                         "        gr.Markdown('Export mode: **' + editor.manifest['mode'] + '**. Selected "
                         "defaults: `' +\n"
                         "                    json.dumps(editor.manifest['validation_best']) + '`. Best "
                         "means lowest validation LPIPS, not perfect.')\n"
                         "        direction = gr.Radio(list(PROMPTS), value='photo_to_pencil', "
                         "label='Direction')\n"
                         "        approach = gr.Dropdown(['validation_best', 'unet', 'gan', 'diffusion'], "
                         "value='validation_best', label='Trained approach')\n"
                         '        with gr.Row():\n'
                         "            source = gr.Image(type='pil', label='Input photo / white-paper pencil "
                         "sketch')\n"
                         "            result = gr.Image(type='pil', format='png', interactive=False, "
                         "label='Actual generated output — preview and download')\n"
                         "        description = gr.Textbox(label='Optional description / desired colours "
                         "(diffusion only)')\n"
                         "        seed = gr.Number(value=42, precision=0, label='Diffusion seed')\n"
                         "        gr.Button('Generate', variant='primary').click(editor, [source, direction, "
                         'approach, seed, description], result, concurrency_limit=1)\n'
                         '        def send_to_photo(image):\n'
                         "            return image, 'sketch_to_photo'\n"
                         "        gr.Button('Use this output as sketch input').click(send_to_photo, result, "
                         '[source, direction])\n'
                         "        gr.Markdown('No original-photo retrieval, saturation trick, or separate "
                         "dark sketch is used. '\n"
                         "                    'Generation is at the training resolution; upscaling is not "
                         "extra learned detail.')\n"
                         "    demo.queue(default_concurrency_limit=1).launch(server_name='127.0.0.1', "
                         'server_port=int(port), show_error=True)\n'}

MODULE_ROOT = Path('/content/sketch2photo_code_face_v1')
for relative, content in MODULE_SOURCES.items():
    destination = MODULE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content)
sys.path.insert(0, str(MODULE_ROOT))
from src.approaches import make_generator, PatchDiscriminator
from src.approaches.diffusion import configure_adapter, noise_prediction_loss
from src.diffusion_backend import BASE_MODEL, BASE_REVISION, PROMPTS, fit_canvas, rgb_on_white, open_pipeline, generate
INFERENCE_SOURCE = MODULE_SOURCES['src/diffusion_backend.py']


In [ ]:
def make_clean_line_art(
    photo_img,
    graphite_strength=1.0,
    background_fade=0.25,
    pencil_texture=0.30,
):
    """
    Produce a white-paper graphite drawing using colour-dodge shading,
    fine dark edges and subtle irregular pencil texture.

    This avoids the embossed appearance of the previous renderer.
    """
    source = rgb_on_white(photo_img)

    working = source.copy()
    working.thumbnail((1600, 1600), Image.Resampling.LANCZOS)

    rgb = np.asarray(working.convert('RGB'))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    scale = max(working.size) / 924.0
    scale = max(scale, 0.35)

    # Preserve important boundaries while reducing digital colour noise.
    smooth = cv2.bilateralFilter(
        gray,
        d=7,
        sigmaColor=32,
        sigmaSpace=max(3, int(round(7 * scale))),
    )

    # Traditional pencil-sketch colour-dodge foundation.
    inverted = 255 - smooth
    blurred_inverted = cv2.GaussianBlur(
        inverted,
        (0, 0),
        sigmaX=max(2.0, 6.0 * scale),
        sigmaY=max(2.0, 6.0 * scale),
    )

    denominator = np.maximum(
        255 - blurred_inverted,
        1,
    ).astype(np.uint8)

    dodge = cv2.divide(
        smooth,
        denominator,
        scale=256,
    ).astype(np.float32)

    dodge = np.clip(dodge, 0, 255)

    # Fine and medium lines: hair, eyes, clothing folds and object edges.
    fine_blur = cv2.GaussianBlur(
        smooth,
        (0, 0),
        sigmaX=max(0.45, 0.65 * scale),
    ).astype(np.float32)

    medium_blur = cv2.GaussianBlur(
        smooth,
        (0, 0),
        sigmaX=max(0.9, 1.65 * scale),
    ).astype(np.float32)

    dark_edges = np.maximum(
        medium_blur - fine_blur,
        0.0,
    ) / 255.0

    # Gentle graphite shading based on original luminance.
    shadow = np.power(
        (255.0 - smooth.astype(np.float32)) / 255.0,
        1.45,
    )

    dodge_density = np.clip(
        (255.0 - dodge) / 255.0,
        0.0,
        1.0,
    )

    density = (
        0.74 * dodge_density
        + 0.22 * shadow
        + 2.10 * dark_edges
    )

    # Fade only smooth, low-detail regions. Do not erase subject edges.
    activity = cv2.GaussianBlur(
        dark_edges,
        (0, 0),
        sigmaX=max(1.5, 4.0 * scale),
    )

    activity_reference = float(
        np.percentile(activity, 88)
    )

    if activity_reference > 1e-6:
        focus = np.clip(
            activity / activity_reference,
            0.0,
            1.0,
        )
    else:
        focus = np.ones_like(activity)

    fade = float(
        np.clip(background_fade, 0.0, 1.0)
    )

    # Keep most tonal information; background fade is intentionally mild.
    density *= (
        1.0
        - 0.30 * fade
        + 0.30 * fade * focus
    )

    strength = float(
        np.clip(graphite_strength, 0.65, 1.60)
    )

    density *= strength

    # Add subtle irregular graphite and directional marks.
    texture = float(
        np.clip(pencil_texture, 0.0, 1.0)
    )

    rng = np.random.default_rng(1947)

    grain = rng.normal(
        0.0,
        1.0,
        gray.shape,
    ).astype(np.float32)

    stroke_length = max(
        3,
        int(round(5 * scale)),
    )

    if stroke_length % 2 == 0:
        stroke_length += 1

    stroke_kernel = np.eye(
        stroke_length,
        dtype=np.float32,
    )

    stroke_kernel /= stroke_kernel.sum()

    directional_marks = cv2.filter2D(
        grain,
        -1,
        stroke_kernel,
        borderType=cv2.BORDER_REFLECT,
    )

    directional_marks *= np.sqrt(stroke_length)

    texture_multiplier = np.clip(
        1.0
        + texture * (
            0.055 * grain
            + 0.085 * directional_marks
        ),
        0.75,
        1.30,
    )

    density *= texture_multiplier
    density = np.clip(density, 0.0, 1.6)

    graphite = 255.0 * (
        1.0 - np.exp(-2.15 * density)
    )

    # Slight paper variation, strongest where graphite is present.
    paper_grain = (
        texture
        * 1.3
        * grain
        * np.sqrt(np.clip(graphite / 255.0, 0.0, 1.0))
    )

    paper = 254.0 + 0.25 * texture * grain

    drawing = np.clip(
        paper - graphite + paper_grain,
        0,
        255,
    ).astype(np.uint8)

    # Prevent pure digital black over large regions.
    drawing = np.maximum(drawing, 8).astype(np.uint8)

    result = Image.fromarray(drawing)

    if result.size != source.size:
        result = result.resize(
            source.size,
            Image.Resampling.LANCZOS,
        )

    return result


# New recommended target settings.
CFG['pencil_strength'] = 0.95
CFG['pencil_background_fade'] = 0.25
CFG['pencil_texture'] = 0.30

print('Natural graphite renderer v6 is ready.')

In [ ]:
def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(data, indent=2, sort_keys=True))
    os.replace(temp, path)

def file_sha(path):
    with open(path, 'rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def download_file(url, destination, attempts=3):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        return destination
    temp = destination.with_name(destination.name + '.part')
    for attempt in range(attempts):
        try:
            with requests.get(url, stream=True, timeout=(20, 90)) as response:
                response.raise_for_status()
                with open(temp, 'wb') as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                expected = response.headers.get('Content-Length')
                if expected is not None and temp.stat().st_size != int(expected):
                    raise IOError('Incomplete download: ' + url)
            os.replace(temp, destination)
            return destination
        except Exception:
            if attempt == attempts - 1:
                raise
            time.sleep(2 * (attempt + 1))

def public_coco_download(relative, destination):
    # Path-style HTTPS access to the same public COCO S3 bucket avoids the custom
    # hostname's certificate mismatch. TLS verification remains enabled.
    return download_file('https://s3.amazonaws.com/images.cocodataset.org/' + relative, destination)

def build_coco_records():
    annotation_zip = public_coco_download('annotations/annotations_trainval2017.zip', CACHE / 'annotations.zip')
    with zipfile.ZipFile(annotation_zip) as archive:
        metadata = json.loads(archive.read('annotations/captions_train2017.json'))
    # Stable master split: pilot is a strict subset of each full split.
    items = sorted((x for x in metadata['images'] if min(x['width'], x['height']) >= 256 and min(x['width'],x['height'])/max(x['width'],x['height']) >= .125), key=lambda x: x['id'])
    random.Random(SEED).shuffle(items)
    assert len(items) >= 64000
    blocks = {'train': items[:60000], 'val': items[60000:62000], 'test': items[62000:64000]}
    records = []
    for split, values in blocks.items():
        for item in values[:CFG[split + '_count']]:
            records.append({'id': str(item['id']), 'group_id': str(item['id']), 'split': split,
                            'photo': str(CACHE / 'photos' / item['file_name']), 'sketch': '',
                            'relative_url': 'train2017/' + item['file_name']})
    def fetch(record):
        path = public_coco_download(record['relative_url'], record['photo'])
        with Image.open(path) as image:
            image.verify()
        record['photo_sha256'] = file_sha(path)
        return record
    with ThreadPoolExecutor(max_workers=8) as pool:
        records = list(tqdm(pool.map(fetch, records), total=len(records), desc='Original COCO photos'))
    # Exact duplicate source bytes must not leak across splits. Drop all members of
    # a duplicate group across splits and report actual remaining counts.
    hash_splits = {}
    for record in records:
        hash_splits.setdefault(record['photo_sha256'], set()).add(record['split'])
    duplicate_hashes = {key for key, values in hash_splits.items() if len(values) > 1}
    if duplicate_hashes:
        print('Removing cross-split exact duplicate groups:', len(duplicate_hashes))
        records = [r for r in records if r['photo_sha256'] not in duplicate_hashes]
    return records

def build_custom_records():
    csv_path = Path(PAIRED_CSV)
    with csv_path.open(newline='') as f:
        rows = list(csv.DictReader(f))
    assert rows and {'photo', 'sketch', 'split', 'group_id'} <= rows[0].keys()
    selected = []
    for split in ['train', 'val', 'test']:
        group = sorted([r for r in rows if r['split'] == split], key=lambda r: (r['group_id'], r['photo']))
        random.Random(SEED).shuffle(group)
        assert group, f'Missing {split} rows in custom CSV'
        selected.extend(group[:CFG[split + '_count']])
    result = []
    for idx, row in enumerate(tqdm(selected, desc='Verify paired files')):
        assert row['group_id'].strip(), 'Every pair needs a group_id for split checks'
        record = {'id': str(idx), 'group_id': row['group_id'], 'split': row['split']}
        sizes = []
        for key in ('photo', 'sketch'):
            path = (csv_path.parent / row[key]).resolve()
            assert path.is_file(), path
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image)
                sizes.append(image.size)
            record[key] = str(path)
            record[key + '_sha256'] = file_sha(path)
        ratios = [w / h for w, h in sizes]
        assert abs(ratios[0] - ratios[1]) < 0.01, f'Pair aspect ratios differ: {record}'
        result.append(record)
    return result


In [ ]:
def build_fs2k_records():
    root = Path(FS2K_ROOT)

    # Download and safely extract FS2K when it is not already available.
    if not (root / 'anno_train.json').exists():
        if not DOWNLOAD_FS2K:
            raise FileNotFoundError(
                'Extract official FS2K to FS2K_ROOT or enable DOWNLOAD_FS2K. '
                'See https://github.com/DengPingFan/FS2K'
            )

        import gdown

        archive = CACHE / 'FS2K.zip'

        if not archive.exists():
            gdown.download(
                id='1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp',
                output=str(archive),
                quiet=False,
            )

        if not zipfile.is_zipfile(archive):
            raise ValueError(
                'FS2K download is not a ZIP. Download the official archive '
                'manually; no fallback was made.'
            )

        extraction = CACHE / 'fs2k_extracted'
        extraction.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(archive) as z:
            extraction_root = extraction.resolve()

            for entry in z.infolist():
                resolved = (extraction / entry.filename).resolve()

                if (
                    not resolved.is_relative_to(extraction_root)
                    or ((entry.external_attr >> 16) & 0o170000) == 0o120000
                ):
                    raise ValueError('Unsafe ZIP entry')

            z.extractall(extraction)

        candidates = list(extraction.rglob('anno_train.json'))

        assert len(candidates) == 1, (
            'Could not find one unique FS2K dataset folder. '
            'Set FS2K_ROOT manually.'
        )

        root = candidates[0].parent

    train_annotation = root / 'anno_train.json'
    test_annotation = root / 'anno_test.json'

    if not test_annotation.exists():
        raise FileNotFoundError(
            f'FS2K test annotation was not found: {test_annotation}'
        )

    train = json.loads(train_annotation.read_text())
    test = json.loads(test_annotation.read_text())

    # Make deterministic train and validation splits, separated by style.
    style_groups = defaultdict(list)

    for row in train:
        style_groups[str(row['style'])].append(row)

    tagged = []

    for style, rows in sorted(style_groups.items()):
        rows = sorted(rows, key=lambda row: row['image_name'])
        random.Random(SEED).shuffle(rows)

        cut = max(1, round(0.2 * len(rows)))

        tagged.extend((row, 'val') for row in rows[:cut])
        tagged.extend((row, 'train') for row in rows[cut:])

    tagged.extend((row, 'test') for row in test)

    # FS2K can contain uppercase or uncommon image extensions.
    # Build an index from the files that actually exist instead of guessing.
    allowed_extensions = {
        '.jpg',
        '.jpeg',
        '.png',
        '.bmp',
        '.webp',
        '.tif',
        '.tiff',
    }

    def normalize_relative_stem(value):
        value = str(value).replace('\\', '/')

        return (
            Path(value)
            .with_suffix('')
            .as_posix()
            .casefold()
        )

    def build_image_index(folder):
        base = root / folder

        if not base.exists():
            raise FileNotFoundError(
                f'FS2K image folder was not found: {base}'
            )

        index = defaultdict(list)

        for path in base.rglob('*'):
            if path.is_file() and path.suffix.lower() in allowed_extensions:
                relative_path = path.relative_to(base)
                key = normalize_relative_stem(relative_path)
                index[key].append(path)

        assert index, f'No FS2K images were found under {base}'

        return index

    photo_index = build_image_index('photo')
    sketch_index = build_image_index('sketch')

    def resolve_image(index, name):
        key = normalize_relative_stem(name)
        matches = index.get(key, [])

        # Return None for both missing and ambiguous paths.
        # We must never guess because that could create incorrect pairs.
        if len(matches) != 1:
            return None

        return matches[0]

    result = []
    excluded = []

    for row, split in tagged:
        stem = str(row['image_name']).replace('\\', '/')
        stem_without_extension = Path(stem).with_suffix('').as_posix()

        # Example:
        # photo3/image0449 -> sketch3/sketch0449
        parts = stem_without_extension.split('/')

        if not parts:
            excluded.append({
                'image_name': stem,
                'split': split,
                'reason': 'invalid annotation path',
                'photo_matches': 0,
                'sketch_matches': 0,
            })
            continue

        parts[0] = parts[0].replace('photo', 'sketch', 1)
        sketch_stem = '/'.join(parts)

        filename = Path(sketch_stem).name
        parent = Path(sketch_stem).parent

        if filename.startswith('image'):
            filename = 'sketch' + filename[len('image'):]

        sketch_stem = (
            filename
            if str(parent) == '.'
            else (parent / filename).as_posix()
        )

        photo_key = normalize_relative_stem(stem_without_extension)
        sketch_key = normalize_relative_stem(sketch_stem)

        photo = resolve_image(photo_index, stem_without_extension)
        sketch = resolve_image(sketch_index, sketch_stem)

        if photo is None or sketch is None:
            excluded.append({
                'image_name': stem,
                'expected_sketch_name': sketch_stem,
                'split': split,
                'reason': 'missing or ambiguous exact path',
                'photo_matches': len(photo_index.get(photo_key, [])),
                'sketch_matches': len(sketch_index.get(sketch_key, [])),
            })
            continue

        # Confirm that paired images have compatible geometry.
        with Image.open(photo) as photo_image:
            photo_width, photo_height = photo_image.size

        with Image.open(sketch) as sketch_image:
            sketch_width, sketch_height = sketch_image.size

        photo_ratio = photo_width / photo_height
        sketch_ratio = sketch_width / sketch_height

        if abs(photo_ratio - sketch_ratio) >= 0.01:
            excluded.append({
                'image_name': stem,
                'expected_sketch_name': sketch_stem,
                'split': split,
                'reason': 'photo and sketch aspect ratios differ',
                'photo_size': [photo_width, photo_height],
                'sketch_size': [sketch_width, sketch_height],
            })
            continue

        result.append({
            'id': 'fs2k_' + stem_without_extension,
            'group_id': 'fs2k_' + stem_without_extension,
            'split': split,
            'domain': 'fs2k_artist',
            'style': str(row['style']),
            'photo': str(photo),
            'sketch': str(sketch),
            'photo_sha256': file_sha(photo),
            'sketch_sha256': file_sha(sketch),
        })

    # Save a report of annotations that could not be paired safely.
    audit = {
        'dataset_root': str(root),
        'annotated_pairs': len(tagged),
        'usable_pairs': len(result),
        'excluded_count': len(excluded),
        'excluded_pairs': excluded,
    }

    atomic_json(
        STUDY_ROOT / 'fs2k_pair_audit.json',
        audit,
    )

    minimum_required = int(0.98 * len(tagged))

    assert len(result) >= minimum_required, (
        f'Too many FS2K pairs are unavailable: '
        f'{len(result)}/{len(tagged)} usable. '
        f'Inspect {STUDY_ROOT / "fs2k_pair_audit.json"}. '
        'The FS2K archive may be incomplete or incorrectly extracted.'
    )

    print(
        'FS2K usable pairs:',
        len(result),
        '/',
        len(tagged),
        '| excluded:',
        len(excluded),
    )

    if excluded:
        print(
            'Incomplete or ambiguous annotations were safely excluded. '
            'Details: fs2k_pair_audit.json'
        )

    # Use a small deterministic subset only in pilot mode.
    if False:  # face rescue uses every audited FS2K pair
        selected = []

        for split in ('train', 'val', 'test'):
            rows = [
                record
                for record in result
                if record['split'] == split
            ]

            random.Random(SEED).shuffle(rows)
            limit = 256 if split == 'train' else 32
            selected.extend(rows[:limit])

        result = selected

        print('Pilot-mode FS2K records:', len(result))

    return result


# The failed mixed-domain run showed that synthetic scenes hide poor real-face quality.
# Use every audited real FS2K pair and do not download COCO in this correction.
RECORDS = build_fs2k_records()
for record in RECORDS:
    record['domain'] = 'fs2k_artist'

# Detect data leakage across train, validation, and test sets.
seen_groups = {}
seen_files = {}

for record in RECORDS:
    group_id = record['group_id']
    split = record['split']

    assert seen_groups.setdefault(group_id, split) == split, (
        f'Group leakage across splits: {group_id}'
    )

    for key in ('photo_sha256', 'sketch_sha256'):
        if key in record:
            file_hash = record[key]

            assert seen_files.setdefault(file_hash, split) == split, (
                f'Identical file crosses splits: {record["id"]}'
            )

SPLITS = {
    split: [
        record
        for record in RECORDS
        if record['split'] == split
    ]
    for split in ('train', 'val', 'test')
}

assert all(SPLITS.values()), (
    'At least one train, validation, or test split is empty.'
)

DATA_FINGERPRINT = hashlib.sha256(
    json.dumps(RECORDS, sort_keys=True).encode()
).hexdigest()

atomic_json(
    STUDY_ROOT / 'data_manifest.json',
    {
        'source': DATA_SOURCE,
        'sha256': DATA_FINGERPRINT,
        'records': RECORDS,
    },
)

print(
    'Actual split/domain counts:',
    Counter(
        (record['split'], record['domain'])
        for record in RECORDS
    ),
)

print(
    'Real drawing style counts:',
    Counter(
        (
            record['split'],
            record.get('style', 'unknown'),
        )
        for record in RECORDS
        if record['sketch']
    ),
)


def training_order(epoch, direction=None):
    order = list(range(len(SPLITS['train'])))
    random.Random(SEED + epoch).shuffle(order)
    return order

VAL_RECORDS = []

for domain in sorted({
    record['domain']
    for record in SPLITS['val']
}):
    rows = [
        record
        for record in SPLITS['val']
        if record['domain'] == domain
    ]

    random.Random(SEED).shuffle(rows)

    VAL_RECORDS += rows[
        :CFG['metric_images_per_domain']
    ]

atomic_json(
    STUDY_ROOT / 'validation_selection_ids.json',
    [record['id'] for record in VAL_RECORDS],
)

print(
    'Checkpoint selection uses',
    len(VAL_RECORDS),
    'fixed validation images with equal domain weighting.',
)

In [ ]:
def uncached_pair(record):
    with Image.open(record['photo']) as image:
        photo, box = fit_canvas(image, CFG['resolution'])
    if record['sketch']:
        with Image.open(record['sketch']) as image:
            sketch, sketch_box = fit_canvas(image, CFG['resolution'])
        assert sketch_box == box, 'Paired geometry must match after fitting'
        sketch = sketch.convert('L').convert('RGB')
    else:
        sketch = make_clean_line_art(photo, CFG['pencil_strength'],
                                    CFG['pencil_background_fade'], CFG['pencil_texture']).convert('RGB')
    return photo, sketch, box


def get_pair(record):
    key = hashlib.sha256((record['photo_sha256']+record.get('sketch_sha256','')+str(CFG['resolution'])+
        str(CFG['pencil_strength'])+str(CFG['pencil_background_fade'])+str(CFG['pencil_texture'])+'v8_face_specific_all_pairs').encode()).hexdigest()
    folder = CACHE/'fitted_pairs'/key[:2]; folder.mkdir(parents=True, exist_ok=True)
    photo_path, sketch_path, box_path = (folder/(key+s) for s in ('_photo.png','_sketch.png','.json'))
    if not box_path.exists():
        photo, sketch, box = uncached_pair(record)
        assert min(box[2]-box[0],box[3]-box[1])>=64, 'Pair aspect ratio is too extreme for shared LPIPS evaluation'
        for image,path in ((photo,photo_path),(sketch,sketch_path)):
            temp=path.with_name(path.name+f'.{os.getpid()}.tmp')
            image.save(temp,format='PNG'); os.replace(temp,path)
        temp=box_path.with_name(box_path.name+f'.{os.getpid()}.tmp')
        temp.write_text(json.dumps(box)); os.replace(temp,box_path)
    with Image.open(photo_path) as p, Image.open(sketch_path) as s:
        return p.convert('RGB'), s.convert('RGB'), tuple(json.loads(box_path.read_text()))

def tensor_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array.transpose(2, 0, 1))

def augment_white_paper_sketch(image, rng):
    """Mild input-only augmentation for real scanner/app pencil variations."""
    sketch = ImageOps.autocontrast(image.convert('L'), cutoff=0.2)
    sketch = ImageEnhance.Contrast(sketch).enhance(rng.uniform(0.85, 1.20))
    sketch = ImageEnhance.Brightness(sketch).enhance(rng.uniform(0.96, 1.04))
    if rng.random() < 0.25:
        sketch = sketch.filter(ImageFilter.MinFilter(3))  # slightly thicker dark strokes
    elif rng.random() < 0.25:
        sketch = sketch.filter(ImageFilter.GaussianBlur(rng.uniform(0.15, 0.55)))
    array = np.asarray(sketch, dtype=np.float32)
    noise = np.random.default_rng(rng.randrange(2**32)).normal(0, rng.uniform(0, 1.5), array.shape)
    return Image.fromarray(np.clip(array + noise, 0, 255).astype(np.uint8)).convert('RGB')


def augment_photo_input(image, rng):
    """Small camera variation without changing geometry or the paired target."""
    image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.96, 1.04))
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.94, 1.06))
    return ImageEnhance.Color(image).enhance(rng.uniform(0.90, 1.10))


class PairedDataset(Dataset):
    def __init__(self, records, direction, training=False, epoch=0):
        self.records, self.direction, self.training, self.epoch = records, direction, training, epoch
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        photo, sketch, _ = get_pair(self.records[index])
        # All randomness is derived from seed/epoch/index, so mid-epoch resume is reproducible.
        rng = random.Random(SEED + 1000003 * self.epoch + index)
        if self.training and rng.random() < 0.5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.direction == 'photo_to_pencil':
            source, target = photo, sketch
            if self.training and rng.random() < CFG['photo_aug_probability']:
                source = augment_photo_input(source, rng)
        else:
            source, target = sketch, photo
            if self.training and rng.random() < CFG['sketch_aug_probability']:
                source = augment_white_paper_sketch(source, rng)
        return {'source': tensor_image(source), 'target': tensor_image(target)}

preview_rows = [
    record
    for domain in sorted({
        record['domain']
        for record in SPLITS['train']
    })
    for record in [
        row
        for row in SPLITS['train']
        if row['domain'] == domain
    ][:6]
]
fig, axes = plt.subplots(len(preview_rows), 2, figsize=(10, 4 * len(preview_rows)))
for i, record in enumerate(preview_rows):
    photo, sketch, box = get_pair(record)
    for ax, im, name in zip(axes[i], [photo.crop(box), sketch.crop(box)], ['Photo target / photo input', 'Pencil target / pencil input']):
        ax.imshow(im); ax.set_title(name); ax.axis('off')
plt.tight_layout()
plt.savefig(RUN_ROOT / 'training_pairs.png', dpi=130)
plt.show()
print('Review these targets before any training. They define the pencil style the models learn.')
if False:  # review is visual but no interactive full-run prompt
    approval = input('Inspect training_pairs.png above. Type TRAIN to start the expensive full run: ').strip()
    if approval != 'TRAIN':
        raise RuntimeError('Training stopped safely before model updates. Fix the data/style, then run again.')
    atomic_json(STUDY_ROOT/'target_review.json', {
        'approved': True, 'typed': approval, 'data_fingerprint': DATA_FINGERPRINT,
        'note': FULL_RUN_REVIEW, 'time_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())})
    print('Target review recorded. Full training is authorized and remains resumable.')


## 2. Review targets and run common metric checks

If the pencil targets are too dark, blurred or stylistically unsuitable, stop here
and improve the paired dataset. Longer training only learns those targets harder.
COCO synthetic and FS2K artist scores are reported separately. Training samples real
pairs at 25% when mixed; every approach uses the same deterministic sampler.
This repeats the small portrait set, so monitor overfitting and style imbalance.

All approaches see RGB tensors in [-1,1], the same white-paper polarity, padding,
resolution and shared paired flip. No dark-sketch inversion is used. Metrics remove
padding; pencil predictions are consistently converted to grayscale. AlexNet LPIPS
v0.1 is the fixed validation selection metric (lower is better); L1, PSNR, SSIM and
colourfulness are also reported. LPIPS is not a complete judge of identity or art.


In [ ]:
PERCEPTUAL = lpips.LPIPS(net='alex', version='0.1').to(DEVICE).eval().requires_grad_(False)

def image_from_tensor(tensor, direction):
    a = ((tensor.detach().float().cpu().clamp(-1,1).permute(1,2,0).numpy()+1)*127.5).round().astype('uint8')
    image = Image.fromarray(a)
    return image.convert('L').convert('RGB') if direction=='photo_to_pencil' else image

@torch.no_grad()
def perceptual_score(pred, target):
    x,y = tensor_image(pred)[None].to(DEVICE), tensor_image(target)[None].to(DEVICE)
    # Extremely narrow photos cannot pass AlexNet's pooling; reject rather than distort them.
    if min(x.shape[-2:])<64: raise ValueError('Cropped test image too narrow for LPIPS; review dataset geometry.')
    return float(PERCEPTUAL(x,y).mean())

@torch.no_grad()
def visual_validation(model, direction, diffusion=False):
    rows = []
    if diffusion:
        previous = next(model.vae.parameters()).dtype
        model.vae.to(dtype=next(model.text_encoder.parameters()).dtype)
        model.unet.eval()
    else: model.eval()
    try:
        for i,r in enumerate(VAL_RECORDS):
            photo,sketch,box = get_pair(r)
            source,target = (photo,sketch) if direction=='photo_to_pencil' else (sketch,photo)
            if diffusion:
                pred = generate(model,source,direction,seed=SEED+i,steps=CFG['inference_steps'],size=CFG['resolution'])
            else:
                pred = image_from_tensor(model(tensor_image(source)[None].to(DEVICE))[0],direction)
            rows.append({'id':r['id'],'domain':r['domain'],'lpips':perceptual_score(pred.crop(box),target.crop(box))})
    finally:
        if diffusion: model.vae.to(dtype=previous); model.unet.train()
        else: model.train()
    grouped = defaultdict(list)
    for r in rows: grouped[r['domain']].append(r['lpips'])
    score = float(np.mean([np.mean(v) for v in grouped.values()]))
    assert math.isfinite(score)
    return score

for direction in PROMPTS:
    sample = PairedDataset(SPLITS['train'][:1],direction)[0]
    assert sample['source'].shape == (3,CFG['resolution'],CFG['resolution'])
    assert all(torch.isfinite(v).all() and v.min()>=-1 and v.max()<=1 for v in sample.values())
photo,sketch,box = get_pair(VAL_RECORDS[0])
assert abs(perceptual_score(photo.crop(box),photo.crop(box)))<1e-5
print('Data ranges, geometry and LPIPS identity check passed.')


## 3. Direction-specific conditional texture refinement

Both generators start from their best completed face-specific checkpoints. Photo→pencil uses
target edges, Laplacian detail, neutral graphite and a low-weight drawing discriminator.
Sketch→photo uses direct RGB, input structure, chroma, colour statistics and a low-weight
photo discriminator. Every validation point is saved so the visual gate can prefer a safe
intermediate checkpoint if the final step becomes too stylized.


In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def atomic_torch(path,state):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp'); torch.save(state,temp); os.replace(temp,path)

def luminance(image):
    return 0.299*image[:,0:1]+0.587*image[:,1:2]+0.114*image[:,2:3]

def chroma_components(image):
    y=luminance(image); return torch.cat((image[:,2:3]-y,image[:,0:1]-y),dim=1)

def spatial_gradients(image):
    return image[:,:,:,1:]-image[:,:,:,:-1],image[:,:,1:,:]-image[:,:,:-1,:]

def colour_statistics(image):
    return image.mean(dim=(2,3)),image.std(dim=(2,3),unbiased=False)

def laplacian(image):
    kernel=image.new_tensor([[0.,1.,0.],[1.,-4.,1.],[0.,1.,0.]])[None,None]
    return F.conv2d(image,kernel.repeat(image.shape[1],1,1,1),padding=1,groups=image.shape[1])

def paired_direction_loss(pred,target,source,direction):
    pred_dx,pred_dy=spatial_gradients(pred); target_dx,target_dy=spatial_gradients(target)
    loss=(CFG['l1_weight_by_direction'][direction]*F.l1_loss(pred,target)+
          CFG['perceptual_weight_by_direction'][direction]*PERCEPTUAL(pred,target).mean()+
          CFG['gradient_weight_by_direction'][direction]*(F.l1_loss(pred_dx,target_dx)+F.l1_loss(pred_dy,target_dy))+
          CFG['laplacian_weight_by_direction'][direction]*F.l1_loss(laplacian(pred),laplacian(target)))
    if direction=='photo_to_pencil':
        neutral=F.l1_loss(pred[:,0:1],pred[:,1:2])+F.l1_loss(pred[:,1:2],pred[:,2:3])
        return loss+CFG['pencil_gray_weight']*neutral
    pred_luma,source_luma=luminance(pred),luminance(source)
    pred_sx,pred_sy=spatial_gradients(pred_luma); source_sx,source_sy=spatial_gradients(source_luma)
    pred_mean,pred_std=colour_statistics(pred); target_mean,target_std=colour_statistics(target)
    return (loss+CFG['structure_weight']*(F.l1_loss(pred_sx,source_sx)+F.l1_loss(pred_sy,source_sy))+
        CFG['chroma_weight']*F.l1_loss(chroma_components(pred),chroma_components(target))+
        CFG['color_stats_weight']*(F.l1_loss(pred_mean,target_mean)+F.l1_loss(pred_std,target_std)))

def train_directional_patchgan(direction,initial_checkpoint):
    seed_everything(SEED); output_mode=OUTPUT_MODES[direction]; target_updates=UPDATE_BUDGETS[direction]
    folder=RUN_ROOT/'unet_patchgan'/direction; folder.mkdir(parents=True,exist_ok=True)
    g_lr=CFG['generator_lr_by_direction'][direction]; d_lr=CFG['discriminator_lr_by_direction'][direction]
    recipe={'direction':direction,'approach':'unet_patchgan','output_mode':output_mode,'config':CFG,
            'data':DATA_FINGERPRINT,'updates':target_updates,'initial_checkpoint':str(initial_checkpoint),
            'implementation':IMPLEMENTATION_SHA}
    signature=hashlib.sha256(json.dumps(recipe,sort_keys=True).encode()).hexdigest()
    atomic_json(folder/'recipe.json',recipe)
    generator=make_generator('unet',CFG['cnn_width'],output_mode=output_mode).to(DEVICE)
    discriminator=PatchDiscriminator(CFG['cnn_width']).to(DEVICE)
    opt_g=torch.optim.AdamW(generator.parameters(),lr=g_lr,weight_decay=CFG['cnn_weight_decay'],betas=(.5,.999))
    opt_d=torch.optim.AdamW(discriminator.parameters(),lr=d_lr,weight_decay=0.,betas=(.5,.999))
    sched_g=torch.optim.lr_scheduler.CosineAnnealingLR(opt_g,T_max=target_updates,eta_min=g_lr*.1)
    sched_d=torch.optim.lr_scheduler.CosineAnnealingLR(opt_d,T_max=target_updates,eta_min=d_lr*.1)
    progress={'step':0,'epoch':0,'offset':0,'best_loss':float('inf'),'best_step':0,'seconds':0.}
    history=[]; last=folder/'last_training_state.pt'
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        assert state['signature']==signature,'Configuration changed; keep this run folder unchanged.'
        generator.load_state_dict(state['generator']); discriminator.load_state_dict(state['discriminator'])
        opt_g.load_state_dict(state['opt_g']); opt_d.load_state_dict(state['opt_d'])
        sched_g.load_state_dict(state['sched_g']); sched_d.load_state_dict(state['sched_d'])
        progress,history=state['progress'],state['history']
        torch.set_rng_state(state['rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng']); del state
        print('Resumed',direction,'at adversarial step',progress['step'])
    else:
        initial_checkpoint=Path(initial_checkpoint)
        state=torch.load(initial_checkpoint,map_location='cpu',weights_only=True)
        assert state['direction']==direction and state.get('output_mode','residual')==output_mode
        assert state['data_fingerprint']==DATA_FINGERPRINT
        generator.load_state_dict(state['generator'],strict=True)
        print('Loaded',direction,'face-specific checkpoint step',state['step']); del state
    started=time.perf_counter(); elapsed_before=progress['seconds']
    def save_last():
        progress['seconds']=elapsed_before+time.perf_counter()-started
        atomic_torch(last,{'signature':signature,'generator':generator.state_dict(),
            'discriminator':discriminator.state_dict(),'opt_g':opt_g.state_dict(),'opt_d':opt_d.state_dict(),
            'sched_g':sched_g.state_dict(),'sched_d':sched_d.state_dict(),'progress':dict(progress),
            'history':history,'rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
            'python_rng':random.getstate(),'numpy_rng':np.random.get_state()})
        atomic_json(folder/'history.json',history)
    def checkpoint(path,score):
        atomic_torch(path,{'generator':{k:v.detach().cpu() for k,v in generator.state_dict().items()},
            'approach':'unet','training':'conditional_patchgan','direction':direction,'output_mode':output_mode,
            'width':CFG['cnn_width'],'resolution':CFG['resolution'],'data_fingerprint':DATA_FINGERPRINT,
            'step':progress['step'],'val_lpips':score,'transferred_from':str(initial_checkpoint)})
    def validate():
        score=visual_validation(generator,direction); history.append({'step':progress['step'],'val_lpips':score})
        if score<progress['best_loss']:
            progress.update(best_loss=score,best_step=progress['step']); checkpoint(folder/'best.pt',score)
        checkpoint(folder/f"snapshot_{progress['step']:07d}.pt",score)
        atomic_json(folder/'selection.json',dict(progress))
    if progress['step']==0 and not history: validate(); save_last()
    generator.train(); discriminator.train()
    bar=tqdm(total=target_updates,initial=progress['step'],desc='U-Net+PatchGAN: '+direction)
    while progress['step']<target_updates:
        order=training_order(progress['epoch'],direction)[progress['offset']:]
        if not order: progress['epoch']+=1; progress['offset']=0; continue
        dataset=PairedDataset(SPLITS['train'],direction,training=True,epoch=progress['epoch'])
        loader=DataLoader(Subset(dataset,order),batch_size=1,num_workers=2,pin_memory=True,
                          generator=torch.Generator().manual_seed(SEED+progress['epoch']))
        for batch in loader:
            source_image=batch['source'].to(DEVICE); target_image=batch['target'].to(DEVICE)
            fake=generator(source_image)
            discriminator.train().requires_grad_(True); opt_d.zero_grad(set_to_none=True)
            real_score=discriminator(torch.cat([source_image,target_image],1))
            fake_score=discriminator(torch.cat([source_image,fake.detach()],1))
            loss_d=F.relu(1-real_score).mean()+F.relu(1+fake_score).mean()
            if not torch.isfinite(loss_d): raise FloatingPointError('Discriminator loss is not finite')
            loss_d.backward(); nn.utils.clip_grad_norm_(discriminator.parameters(),1.); opt_d.step(); sched_d.step()
            discriminator.eval().requires_grad_(False); opt_g.zero_grad(set_to_none=True)
            paired=paired_direction_loss(fake,target_image,source_image,direction)
            adversarial=-discriminator(torch.cat([source_image,fake],1)).mean()
            loss_g=paired+CFG['adversarial_weight_by_direction'][direction]*adversarial
            if not torch.isfinite(loss_g): raise FloatingPointError('Generator loss is not finite')
            loss_g.backward(); nn.utils.clip_grad_norm_(generator.parameters(),1.,error_if_nonfinite=True)
            opt_g.step(); sched_g.step(); discriminator.train().requires_grad_(True)
            progress['step']+=1; progress['offset']+=1; step=progress['step']
            history.append({'step':step,'train_loss':float(loss_g.detach()),'paired_loss':float(paired.detach()),
                'adversarial_loss':float(adversarial.detach()),'discriminator_loss':float(loss_d.detach()),
                'lr':sched_g.get_last_lr()[0]})
            bar.update(1); bar.set_postfix(g=round(float(loss_g.detach()),3),d=round(float(loss_d.detach()),3))
            if step%CFG['cnn_validate_every']==0 or step==target_updates: validate()
            if step%CFG['cnn_save_every']==0 or step==target_updates: save_last()
            if step==target_updates: break
        del loader
    bar.close(); save_last()
    atomic_json(folder/'compute.json',{'generator_trainable_parameters':sum(p.numel() for p in generator.parameters()),
        'discriminator_trainable_parameters':sum(p.numel() for p in discriminator.parameters()),
        'training_and_validation_seconds':progress['seconds'],'hardware':atomic_environment,
        'best_step':progress['best_step'],'completed_updates':progress['step']})
    del generator,discriminator,opt_g,opt_d,sched_g,sched_d; gc.collect(); torch.cuda.empty_cache()
    return dict(progress)


## 4. Approach C — pretrained diffusion fine-tuning

Attention and the input convolution receive rank-32 LoRA adapters. The eight-channel
input layer is preserved. Source VAE latents are unscaled; target VAE latents use the
VAE scaling factor, matching InstructPix2Pix training/inference. Base weights, VAE
and text encoder are frozen. Min-SNR diffusion noise loss, conditioning dropout,
gradient accumulation, FP32 VAE and FP16 scaler/backoff are used.

Selection now uses generated-image LPIPS on the SAME fixed validation images as A/B,
not denoising loss alone. Both loss curves are logged. Step zero is the pretrained
baseline: if it stays best, explicitly report that fine-tuning did not improve it.
The full budget is five epochs with early stopping, not the previous one-epoch default.


In [ ]:
def preview_pipeline(pipe, direction, folder, records=None, title=''):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    if records is None:
        domains = sorted({record['domain'] for record in SPLITS['val']})
        records = [next(record for record in SPLITS['val'] if record['domain'] == domain) for domain in domains]
        records = records[:max(CFG['validation_preview_images'], len(domains))]
    # Training encodes with FP32 VAE; the inference pipeline supplies inputs in its model dtype.
    previous_vae_dtype = next(pipe.vae.parameters()).dtype
    pipe.vae.to(dtype=next(pipe.text_encoder.parameters()).dtype)
    fig, axes = plt.subplots(len(records), 3, figsize=(12, 4 * len(records)), squeeze=False)
    for i, record in enumerate(records):
        photo, sketch, box = get_pair(record)
        source, target = (photo, sketch) if direction == 'photo_to_pencil' else (sketch, photo)
        # Already fitted canvas: generate returns the full square; then crop the shared box.
        generated = generate(pipe, source, direction, seed=SEED+i, steps=CFG['inference_steps'], size=CFG['resolution'])
        generated.save(folder / f'prediction_{i}.png')
        for ax, im, label in zip(axes[i], [source, generated, target], ['Input', title, 'Training target']):
            ax.imshow(im.crop(box)); ax.set_title(label); ax.axis('off')
    plt.tight_layout(); plt.savefig(folder / 'comparison.png', dpi=140); plt.show(); plt.close(fig)
    pipe.vae.to(dtype=previous_vae_dtype)


In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def create_training_components(direction):
    seed_everything(SEED)
    pipe = open_pipeline(device=DEVICE, offload=False)
    configure_adapter(pipe, CFG['lora_rank'])
    with torch.no_grad():
        ids = pipe.tokenizer([PROMPTS[direction], ''], padding='max_length', truncation=True,
                             max_length=pipe.tokenizer.model_max_length, return_tensors='pt').input_ids.to(DEVICE)
        embeddings = pipe.text_encoder(ids)[0].detach()
    noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, revision=BASE_REVISION, subfolder='scheduler')
    params = [p for p in pipe.unet.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=CFG['learning_rate'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min(1.0, (step + 1) / max(1, CFG['warmup_updates'])))
    scaler = torch.amp.GradScaler('cuda', init_scale=1024.0)
    print('Trainable parameters:', sum(p.numel() for p in params))
    return pipe, noise_scheduler, embeddings, params, optimizer, scheduler, scaler

def diffusion_loss(pipe, noise_scheduler, embeddings, batch, training=True, fixed_seed=None, timestep=None):
    return noise_prediction_loss(pipe, noise_scheduler, embeddings, batch, device=DEVICE, dtype=DTYPE,
        conditioning_dropout=CFG['conditioning_dropout'], snr_gamma=CFG['snr_gamma'],
        training=training, fixed_seed=fixed_seed, timestep=timestep)

@torch.no_grad()
def validation_loss(pipe, noise_scheduler, embeddings, direction):
    pipe.unet.eval()
    dataset = PairedDataset(SPLITS['val'][:CFG['validation_loss_images']], direction)
    values = []
    for i in range(len(dataset)):
        batch = {k: v.unsqueeze(0) for k,v in dataset[i].items()}
        for timestep in (100, 300, 500, 700):
            values.append(float(diffusion_loss(pipe, noise_scheduler, embeddings, batch,
                                               training=False, fixed_seed=SEED+i, timestep=timestep)))
    pipe.unet.train()
    return float(np.mean(values))

def save_best(pipe, folder, metadata):
    root = Path(folder).parent
    snapshot = root / ('best_step_' + str(metadata['step']).zfill(7))
    snapshot.mkdir(parents=True, exist_ok=True)
    pipe.unet.save_pretrained(snapshot, safe_serialization=True)
    atomic_json(snapshot / 'selection.json', metadata)
    # Commit the pointer only after all files exist. Previous best snapshots remain recoverable.
    atomic_json(root / 'best_pointer.json', {'folder': snapshot.name})

def best_path(direction):
    root = RUN_ROOT / direction
    return root / json.loads((root / 'best_pointer.json').read_text())['folder']

def save_resume(path, pipe, optimizer, scheduler, scaler, progress, signature, history):
    # Load this pickle only from your own training run. Exports use safetensors for inference.
    state = {'adapter': {k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(pipe.unet).items()},
             'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
             'scaler': scaler.state_dict(), 'progress': progress, 'signature': signature,
             'history': history, 'torch_rng': torch.get_rng_state(),
             'cuda_rng': torch.cuda.get_rng_state_all(), 'numpy_rng': np.random.get_state(),
             'python_rng': random.getstate()}
    temp = Path(str(path) + '.tmp')
    torch.save(state, temp); os.replace(temp, path)

def train_direction(direction):
    folder = RUN_ROOT / direction
    folder.mkdir(parents=True, exist_ok=True)
    recipe = {'config': CFG, 'data': DATA_FINGERPRINT, 'source': DATA_SOURCE,
              'direction': direction, 'base': BASE_MODEL, 'revision': BASE_REVISION,
              'prompt': PROMPTS[direction], 'algorithm': 'ip2p-lora-study-v2',
              'implementation': IMPLEMENTATION_SHA}
    signature = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()
    recipe_path = folder / 'recipe.json'
    if recipe_path.exists():
        assert json.loads(recipe_path.read_text()) == recipe, 'Configuration changed. Use a new EXPERIMENT name.'
    else:
        atomic_json(recipe_path, recipe)
    pipe, noise_scheduler, embeddings, params, optimizer, scheduler, scaler = create_training_components(direction)
    progress = {'step': 0, 'epoch': 0, 'offset': 0, 'best_loss': float('inf'), 'best_step': 0, 'stale': 0, 'seconds': 0.}
    history = []
    resume = folder / 'last_training_state.pt'
    if resume.exists():
        state = torch.load(resume, map_location='cpu', weights_only=False)
        assert state['signature'] == signature, 'Resume signature mismatch'
        set_peft_model_state_dict(pipe.unet, state['adapter'])
        optimizer.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        progress, history = state['progress'], state['history']
        torch.set_rng_state(state['torch_rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
        np.random.set_state(state['numpy_rng']); random.setstate(state['python_rng'])
        del state
        print('Resuming', direction, progress)
    else:
        baseline_loss = visual_validation(pipe, direction, diffusion=True)
        assert math.isfinite(baseline_loss)
        progress['best_loss'] = baseline_loss
        history.append({'step': 0, 'val_lpips': baseline_loss, 'label': 'pretrained'})
        save_best(pipe, folder / 'best', {'step': 0, 'val_lpips': baseline_loss, 'improved': False})

    started = time.perf_counter(); elapsed_before = progress['seconds']
    accumulation = CFG['gradient_accumulation']
    per_epoch = math.ceil(len(SPLITS['train']) / accumulation)
    target_steps = CFG['pilot_updates'] if MODE == 'pilot' else round(CFG['full_epochs'] * per_epoch)
    atomic_json(folder / 'budget.json', {'target_updates': target_steps, 'train_pairs': len(SPLITS['train']),
                                       'effective_batch': accumulation, 'mode': MODE})
    print(direction, ':', target_steps, 'updates; effective batch', accumulation)
    pipe.unet.train()
    torch.cuda.reset_peak_memory_stats()
    timing = []
    bar = tqdm(total=target_steps, initial=progress['step'], desc=direction)
    while progress['step'] < target_steps and progress['stale'] < CFG['early_stop_patience']:
        epoch = progress['epoch']
        dataset = PairedDataset(SPLITS['train'], direction, training=True, epoch=epoch)
        order = training_order(epoch,direction)
        remaining = order[progress['offset']:]
        if not remaining:
            progress['epoch'] += 1; progress['offset'] = 0
            continue
        loader = DataLoader(Subset(dataset, remaining), batch_size=1, shuffle=False, num_workers=2,
                            pin_memory=True, generator=torch.Generator().manual_seed(SEED+epoch))
        iterator = iter(loader)
        batches_remaining = len(loader)
        while batches_remaining and progress['step'] < target_steps and progress['stale'] < CFG['early_stop_patience']:
            count = min(accumulation, batches_remaining)
            torch.cuda.synchronize(); start = time.perf_counter()
            window = [next(iterator) for _ in range(count)]
            for retry in range(6):
                optimizer.zero_grad(set_to_none=True)
                running_loss = 0.0
                for batch in window:
                    loss = diffusion_loss(pipe, noise_scheduler, embeddings, batch)
                    if not torch.isfinite(loss):
                        raise FloatingPointError('Non-finite loss; stop and inspect inputs/mixed precision.')
                    running_loss += float(loss.detach())
                    scaler.scale(loss / count).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale:
                    break
                print('Retrying this accumulation window after FP16 scale backoff:', scaler.get_scale())
            else:
                raise FloatingPointError('Repeated FP16 overflow after scale backoff; last saved checkpoint is recoverable.')
            del window
            scheduler.step()
            progress['step'] += 1; progress['offset'] += count
            batches_remaining -= count
            torch.cuda.synchronize(); timing.append(time.perf_counter() - start)
            step = progress['step']
            history.append({'step': step, 'train_loss': running_loss/count, 'lr': scheduler.get_last_lr()[0],
                            'seconds': timing[-1]})
            bar.update(1); bar.set_postfix(loss=round(running_loss/count, 4))
            if len(timing) == 10:
                seconds = float(np.median(timing[2:]))
                full_steps = math.ceil(60000 / accumulation) * CFG['full_epochs']
                estimate = {'seconds_per_update': seconds, 'estimated_60k_hours_per_direction': seconds*full_steps/3600,
                            'peak_vram_gib': torch.cuda.max_memory_allocated()/2**30,
                            'excludes': 'downloads, validation, checkpoint IO and runtime interruptions'}
                atomic_json(folder / 'speed_estimate.json', estimate)
                print('Measured estimate:', estimate)
            if step % CFG['validate_every'] == 0 or step == target_steps:
                noise_score = validation_loss(pipe, noise_scheduler, embeddings, direction)
                score = visual_validation(pipe, direction, diffusion=True)
                assert math.isfinite(score)
                history[-1]['val_noise_loss'] = noise_score
                history[-1]['val_lpips'] = score
                if score < progress['best_loss']:
                    substantial = score < progress['best_loss'] - CFG['min_delta']
                    progress['stale'] = 0 if substantial else progress['stale'] + 1
                    progress['best_loss'], progress['best_step'] = score, step
                    save_best(pipe, folder / 'best', {'step': step, 'val_lpips': score, 'improved': True})
                else:
                    progress['stale'] += 1
                # Same fixed validation sources and seeds; never inspect held-out test for tuning.
                pipe.unet.eval()
                preview_pipeline(pipe, direction, folder / f'validation_step_{step}', title=f'Fine-tuned step {step}')
                pipe.unet.train()
            if step % CFG['save_every'] == 0 or step == target_steps or progress['stale'] >= CFG['early_stop_patience']:
                progress['seconds'] = elapsed_before + time.perf_counter() - started
                save_resume(resume, pipe, optimizer, scheduler, scaler, dict(progress), signature, history)
                atomic_json(folder / 'history.json', history)
        del loader, iterator
    bar.close()
    print('Best validation step:', progress['best_step'], 'loss:', progress['best_loss'])
    if progress['best_step'] == 0:
        print('No validation improvement over pretrained initialization. Do not claim training improved quality.')
    atomic_json(folder / 'compute.json', {'generator_trainable_parameters': sum(p.numel() for p in params),
        'discriminator_trainable_parameters': 0, 'training_and_validation_seconds': progress['seconds'],
        'hardware': atomic_environment, 'best_step': progress['best_step'], 'completed_updates': progress['step']})
    del pipe, optimizer, scheduler, scaler, params, embeddings, noise_scheduler
    gc.collect(); torch.cuda.empty_cache()
    return progress


## 5. Refine both face directions

This cell performs two resumable 4,096-update runs. Each begins from the completed
face-specific U-Net and adds only direction-appropriate conditional texture supervision.
Validation snapshots are saved every 512 updates; the previous models are not overwritten.


In [ ]:
# Training/evaluation safety-filter recovery.
# Keep the safety checker enabled. Retry deterministic alternate seeds when
# a benign validation image is falsely filtered. If every retry is filtered,
# return a black failure image so evaluation penalizes it instead of crashing.

import warnings
from PIL import Image

if '_ORIGINAL_SAFE_GENERATE' not in globals():
    _ORIGINAL_SAFE_GENERATE = generate

SAFETY_FILTER_EVENTS = []


def generate(
    pipe,
    image,
    direction,
    seed=42,
    steps=30,
    text_guidance=7.0,
    image_guidance=1.5,
    size=512,
    description='',
):
    retry_seeds = (
        int(seed),
        int(seed) + 1009,
        int(seed) + 2017,
        int(seed) + 4093,
    )

    last_error = None

    for attempt, retry_seed in enumerate(retry_seeds):
        try:
            result = _ORIGINAL_SAFE_GENERATE(
                pipe=pipe,
                image=image,
                direction=direction,
                seed=retry_seed,
                steps=steps,
                text_guidance=text_guidance,
                image_guidance=image_guidance,
                size=size,
                description=description,
            )

            if attempt > 0:
                SAFETY_FILTER_EVENTS.append({
                    'direction': direction,
                    'requested_seed': int(seed),
                    'used_seed': retry_seed,
                    'attempt': attempt + 1,
                    'status': 'retry_succeeded',
                })

                print(
                    'Safety checker retry succeeded:',
                    direction,
                    'requested seed',
                    seed,
                    'used seed',
                    retry_seed,
                )

            return result

        except RuntimeError as error:
            message = str(error)

            if 'filtered this output' not in message:
                raise

            last_error = error

            SAFETY_FILTER_EVENTS.append({
                'direction': direction,
                'requested_seed': int(seed),
                'used_seed': retry_seed,
                'attempt': attempt + 1,
                'status': 'filtered',
            })

            print(
                'Safety checker filtered validation output; retrying:',
                direction,
                'seed',
                retry_seed,
            )

    # Every deterministic retry was filtered.
    # Return a black failure prediction so LPIPS penalizes it.
    _, box = fit_canvas(image, int(size))

    width = box[2] - box[0]
    height = box[3] - box[1]

    warnings.warn(
        f'All safety-checker retries were filtered for {direction}. '
        'A black failure prediction is being used for evaluation.',
        RuntimeWarning,
    )

    failure = Image.new(
        'RGB',
        (width, height),
        'black',
    )

    if direction == 'photo_to_pencil':
        failure = failure.convert('L').convert('RGB')

    return failure


gc.collect()
torch.cuda.empty_cache()

print(
    'Safety-filter retry wrapper installed. '
    'The safety checker remains enabled.'
)

In [ ]:
PREVIOUS_SELECTED=json.loads((PREVIOUS_ROOT/'selected_models.json').read_text())
SELECTED={}
for direction in PROMPTS:
    initial=Path(PREVIOUS_SELECTED[direction]['unet']['path'])
    result=train_directional_patchgan(direction,initial)
    folder=STUDY_ROOT/'unet_patchgan'/direction
    SELECTED[direction]={'unet':{'folder':str(folder),'path':str(folder/'best.pt'),
        'val_lpips':result['best_loss'],'best_step':result['best_step'],'tuned':True,
        'output_mode':OUTPUT_MODES[direction],'training':'conditional_patchgan',
        'warm_started_from':str(initial)}}
atomic_json(STUDY_ROOT/'selected_models.json',SELECTED)
print('Dual-direction refinement complete:',json.dumps(SELECTED,indent=2))


## 6. Saved dual-direction candidates

Confirm both short runs completed. Their validation snapshots and the earlier face-specific
checkpoints remain available for visual selection.


In [ ]:
print(json.dumps(SELECTED,indent=2))
for direction in PROMPTS:
    assert (Path(SELECTED[direction]['unet']['folder'])/'snapshot_0004096.pt').is_file()
print('Both final snapshots exist. Continue to the visual gate; do not run Section 8.')


## 7. Final dual-direction visual gate

For four unseen faces in each direction, compare the previous face-specific model, the
LPIPS-selected adversarial checkpoint, and the final 4,096-step snapshot. This prevents a
smooth LPIPS winner or an over-stylized final checkpoint from being accepted blindly.


In [ ]:
def load_unet_checkpoint(path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    model=make_generator('unet',state['width'],output_mode=state.get('output_mode','residual')).to(DEVICE)
    model.load_state_dict(state['generator'],strict=True); return model.eval(),state

@torch.no_grad()
def predict_unet(model,source,direction):
    return image_from_tensor(model(tensor_image(source)[None].to(DEVICE))[0],direction)

REPORTS=STUDY_ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
examples=VAL_RECORDS[:min(4,len(VAL_RECORDS))]
for direction in PROMPTS:
    previous,previous_state=load_unet_checkpoint(PREVIOUS_SELECTED[direction]['unet']['path'])
    selected,selected_state=load_unet_checkpoint(SELECTED[direction]['unet']['path'])
    final_path=Path(SELECTED[direction]['unet']['folder'])/'snapshot_0004096.pt'
    final_model,final_state=load_unet_checkpoint(final_path)
    fig,axes=plt.subplots(len(examples),5,figsize=(18,4*len(examples)),squeeze=False)
    for index,record in enumerate(examples):
        photo,sketch,box=get_pair(record)
        source_image,target_image=(photo,sketch) if direction=='photo_to_pencil' else (sketch,photo)
        images=[source_image.crop(box),target_image.crop(box),
            predict_unet(previous,source_image,direction).crop(box),
            predict_unet(selected,source_image,direction).crop(box),
            predict_unet(final_model,source_image,direction).crop(box)]
        labels=['Input','Real target','Before PatchGAN',
            'LPIPS-selected step '+str(selected_state['step']),'Final step 4096']
        for axis,image,label in zip(axes[index],images,labels):
            axis.imshow(image); axis.set_title(label); axis.axis('off')
    fig.suptitle(direction+' final visual gate')
    fig.tight_layout(); fig.savefig(REPORTS/(direction+'_dual_patchgan_gate.png'),dpi=150); plt.show(); plt.close(fig)
    history=json.loads((Path(SELECTED[direction]['unet']['folder'])/'history.json').read_text())
    fig,axes=plt.subplots(1,2,figsize=(12,3))
    for axis,key,title in zip(axes,('val_lpips','discriminator_loss'),('Validation LPIPS','Discriminator loss')):
        points=[(row['step'],row[key]) for row in history if key in row]
        if points: axis.plot(*zip(*points)); axis.set_title(title); axis.set_xlabel('Updates')
    fig.tight_layout(); plt.show(); plt.close(fig)
    del previous,selected,final_model; gc.collect(); torch.cuda.empty_cache()

print('FINAL VISUAL GATE: inspect both directions and all four unseen faces.')
print('Reject identity drift, blur, washed-out pencil, sepia/monochrome photo, or artifacts.')
print('Do not run Section 8 until this comparison is reviewed.')


## 8. Stage B - persistent 20K bidirectional general-image refinement

The face gate is not sufficient evidence for a general Sketch2Photo application.
This stage therefore refines **both directions** on the same deterministic 20,000-image
COCO training subset, while replaying every FS2K training portrait twice to reduce
face forgetting. The reviewed face checkpoints remain untouched as fallbacks.

The exact 20,000 COCO training IDs are fixed by the original master split. Images are
stored under Google Drive, so interrupted downloads and either direction can resume
without restarting. Each direction sees all 20,000 general images. The weak
pencil-to-photo direction runs first; photo-to-pencil follows. The first 12,000 updates
of each direction use 256px for speed and the remaining updates use 512px for detail.
Stop after the general visual gate and review all four figures before testing.


In [ ]:
GENERAL_EXPERIMENT = 'general_20k_bidirectional_v1'
GENERAL_ROOT = SAVE_ROOT / GENERAL_EXPERIMENT / 'general_rescue'
GENERAL_DATA_ROOT = SAVE_ROOT / 'datasets' / 'coco2017_general_20k'
GENERAL_PHOTO_ROOT = GENERAL_DATA_ROOT / 'photos'
GENERAL_ROOT.mkdir(parents=True, exist_ok=True)
GENERAL_PHOTO_ROOT.mkdir(parents=True, exist_ok=True)

GENERAL_CFG = {
    'train_count': 20000,
    'val_count': 256,
    'test_count': 256,
    'fs2k_replays': 2,
    'progressive_512_start': 12000,
    'validate_every': 2000,
    'save_every': 500,
    'generator_lr_by_direction': {
        'sketch_to_photo': 2e-5,
        'photo_to_pencil': 1e-5,
    },
    'discriminator_lr': 1e-4,
    'weight_decay': 1e-4,
    'adversarial_max_by_direction': {
        'sketch_to_photo': 1.0,
        'photo_to_pencil': .5,
    },
    'adversarial_warmup': 3000,
    'validation_coco': 16,
    'validation_fs2k': 16,
    'direction_order': ['sketch_to_photo', 'photo_to_pencil'],
}

assert SELECTED['photo_to_pencil']['unet']['best_step'] == 2560, (
    'Expected the reviewed photo-to-pencil checkpoint at step 2560.'
)
assert SELECTED['sketch_to_photo']['unet']['best_step'] == 0, (
    'Expected the failed sketch-to-photo PatchGAN to fall back to step 0.'
)

print('Photo-to-pencil fallback:', SELECTED['photo_to_pencil']['unet']['path'])
print('Sketch-to-photo fallback:', SELECTED['sketch_to_photo']['unet']['path'])
print('Persistent COCO subset:', GENERAL_DATA_ROOT)
print('This stage downloads 20,512 COCO images once (20K train + fixed validation/test).')
print('Both directions train on all 20,000 general images; either can resume independently.')


In [ ]:
def build_persistent_general_coco_records():
    annotation_zip = public_coco_download(
        'annotations/annotations_trainval2017.zip',
        GENERAL_DATA_ROOT / 'annotations_trainval2017.zip',
    )
    with zipfile.ZipFile(annotation_zip) as archive:
        metadata = json.loads(archive.read('annotations/captions_train2017.json'))

    items = sorted(
        (
            item for item in metadata['images']
            if min(item['width'], item['height']) >= 256
            and min(item['width'], item['height']) / max(item['width'], item['height']) >= .125
        ),
        key=lambda item: item['id'],
    )
    random.Random(SEED).shuffle(items)
    assert len(items) >= 64000

    # These boundaries are identical to the original master experiment. Validation
    # and test never overlap the first 60K training pool.
    blocks = {
        'train': items[:GENERAL_CFG['train_count']],
        'val': items[60000:60000 + GENERAL_CFG['val_count']],
        'test': items[62000:62000 + GENERAL_CFG['test_count']],
    }
    records = []
    for split, rows in blocks.items():
        for item in rows:
            records.append({
                'id': 'coco_' + str(item['id']),
                'group_id': 'coco_' + str(item['id']),
                'split': split,
                'domain': 'coco_synthetic',
                'photo': str(GENERAL_PHOTO_ROOT / item['file_name']),
                'sketch': '',
                'relative_url': 'train2017/' + item['file_name'],
            })

    def fetch(record):
        path = public_coco_download(record['relative_url'], record['photo'])
        with Image.open(path) as image:
            image.verify()
        record['photo_sha256'] = file_sha(path)
        return record

    with ThreadPoolExecutor(max_workers=8) as pool:
        records = list(tqdm(
            pool.map(fetch, records),
            total=len(records),
            desc='Persistent COCO 20K + validation/test',
        ))

    # Prove that no exact source file crosses a split.
    seen = {}
    for record in records:
        digest = record['photo_sha256']
        previous = seen.setdefault(digest, record['split'])
        assert previous == record['split'], 'Exact COCO image crosses data splits.'

    manifest = {
        'source': 'COCO 2017 captions image metadata',
        'seed': SEED,
        'split_rule': 'shuffled eligible IDs: train[0:20000], val[60000:60256], test[62000:62256]',
        'counts': {split: len(rows) for split, rows in blocks.items()},
        'records': records,
    }
    atomic_json(GENERAL_DATA_ROOT / 'manifest.json', manifest)
    print('Persistent COCO counts:', Counter(record['split'] for record in records))
    return records


GENERAL_COCO_RECORDS = build_persistent_general_coco_records()
GENERAL_COCO_SPLITS = {
    split: [record for record in GENERAL_COCO_RECORDS if record['split'] == split]
    for split in ('train', 'val', 'test')
}

# Every COCO training image appears once. Real FS2K pairs are replayed twice to
# prevent the broad synthetic stage from erasing the portrait knowledge.
GENERAL_TRAIN_RECORDS = list(GENERAL_COCO_SPLITS['train'])
for _ in range(GENERAL_CFG['fs2k_replays']):
    GENERAL_TRAIN_RECORDS.extend(dict(record) for record in SPLITS['train'])

random.Random(SEED + 700).shuffle(GENERAL_TRAIN_RECORDS)
GENERAL_VAL_RECORDS = (
    GENERAL_COCO_SPLITS['val'][:GENERAL_CFG['validation_coco']]
    + SPLITS['val'][:GENERAL_CFG['validation_fs2k']]
)

GENERAL_DATA_FINGERPRINT = hashlib.sha256(json.dumps({
    'coco': [(record['id'], record['split'], record['photo_sha256']) for record in GENERAL_COCO_RECORDS],
    'fs2k': DATA_FINGERPRINT,
    'replays': GENERAL_CFG['fs2k_replays'],
}, sort_keys=True).encode()).hexdigest()

atomic_json(GENERAL_ROOT / 'training_manifest.json', {
    'unique_coco_train_images': len(GENERAL_COCO_SPLITS['train']),
    'unique_fs2k_train_pairs': len(SPLITS['train']),
    'total_updates': len(GENERAL_TRAIN_RECORDS),
    'validation_coco': len(GENERAL_COCO_SPLITS['val']),
    'test_coco': len(GENERAL_COCO_SPLITS['test']),
    'fingerprint': GENERAL_DATA_FINGERPRINT,
})
print('General training updates:', len(GENERAL_TRAIN_RECORDS))
print('Unique training sources:', len(GENERAL_COCO_SPLITS['train']) + len(SPLITS['train']))


In [ ]:
def get_general_pair(record, resolution=512):
    if record['domain'] == 'fs2k_artist':
        photo, sketch, _ = get_pair(record)
        if photo.size != (resolution, resolution):
            photo = photo.resize((resolution, resolution), Image.Resampling.LANCZOS)
            sketch = sketch.resize((resolution, resolution), Image.Resampling.LANCZOS)
        return photo, sketch

    with Image.open(record['photo']) as image:
        photo, _ = fit_canvas(image, resolution)
    sketch = make_clean_line_art(
        photo,
        CFG['pencil_strength'],
        CFG['pencil_background_fade'],
        CFG['pencil_texture'],
    ).convert('RGB')
    return photo, sketch


class GeneralBidirectionalDataset(Dataset):
    def __init__(self, records, direction, resolution, training=True):
        self.records = records
        self.direction = direction
        self.resolution = resolution
        self.training = training

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        photo, sketch = get_general_pair(record, self.resolution)
        rng = random.Random(SEED + 1000003 * index)
        if self.training and rng.random() < .5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.training and rng.random() < .35:
            sketch = augment_white_paper_sketch(sketch, rng)
        source_image, target_image = (
            (photo, sketch) if self.direction == 'photo_to_pencil' else (sketch, photo)
        )
        return {
            'source': tensor_image(source_image),
            'target': tensor_image(target_image),
            'domain': record['domain'],
        }


def general_sketch_photo_loss(prediction, target, source):
    pred_dx, pred_dy = spatial_gradients(prediction)
    target_dx, target_dy = spatial_gradients(target)
    pred_luma, source_luma = luminance(prediction), luminance(source)
    pred_sx, pred_sy = spatial_gradients(pred_luma)
    source_sx, source_sy = spatial_gradients(source_luma)
    pred_mean, pred_std = colour_statistics(prediction)
    target_mean, target_std = colour_statistics(target)
    return (
        10.0 * F.l1_loss(prediction, target)
        + 4.0 * PERCEPTUAL(prediction, target).mean()
        + 4.0 * (F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy))
        + 2.0 * F.l1_loss(laplacian(prediction), laplacian(target))
        + 0.75 * (F.l1_loss(pred_sx, source_sx) + F.l1_loss(pred_sy, source_sy))
        + 2.0 * F.l1_loss(chroma_components(prediction), chroma_components(target))
        + 1.0 * (F.l1_loss(pred_mean, target_mean) + F.l1_loss(pred_std, target_std))
    )


@torch.no_grad()
def general_validation(model, direction):
    model.eval()
    rows = []
    for record in GENERAL_VAL_RECORDS:
        photo, sketch = get_general_pair(record, CFG['resolution'])
        source_image, target_image = (
            (photo, sketch) if direction == 'photo_to_pencil' else (sketch, photo)
        )
        source_tensor = tensor_image(source_image)[None].to(DEVICE)
        target_tensor = tensor_image(target_image)[None].to(DEVICE)
        prediction = model(source_tensor)
        rows.append({
            'id': record['id'],
            'domain': record['domain'],
            'direction': direction,
            'lpips': float(PERCEPTUAL(prediction, target_tensor).mean()),
            'mae': float(F.l1_loss(prediction, target_tensor) / 2),
        })
    model.train()
    return rows


def general_direction_loss(prediction, target, source, direction):
    if direction == 'photo_to_pencil':
        return paired_direction_loss(prediction, target, source, direction)
    return general_sketch_photo_loss(prediction, target, source)

def train_general_direction(direction, initial_checkpoint):
    seed_everything(SEED)
    folder = GENERAL_ROOT / 'unet_multiscale_patchgan' / direction
    folder.mkdir(parents=True, exist_ok=True)
    target_updates = len(GENERAL_TRAIN_RECORDS)
    initial_checkpoint = Path(initial_checkpoint)
    recipe = {
        'direction': direction,
        'approach': 'direct_unet_multiscale_patchgan',
        'general_config': GENERAL_CFG,
        'target_updates': target_updates,
        'initial_checkpoint': str(initial_checkpoint),
        'data_fingerprint': GENERAL_DATA_FINGERPRINT,
        'implementation': IMPLEMENTATION_SHA + '-general20k-v1',
    }
    signature = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()
    atomic_json(folder / 'recipe.json', recipe)

    output_mode = OUTPUT_MODES[direction]
    generator = make_generator('unet', CFG['cnn_width'], output_mode=output_mode).to(DEVICE)
    initial = torch.load(initial_checkpoint, map_location='cpu', weights_only=True)
    assert initial['direction'] == direction
    assert initial.get('output_mode', 'residual') == output_mode
    generator.load_state_dict(initial['generator'], strict=True)
    discriminator_full = PatchDiscriminator(CFG['cnn_width']).to(DEVICE)
    discriminator_half = PatchDiscriminator(CFG['cnn_width']).to(DEVICE)

    opt_g = torch.optim.AdamW(
        generator.parameters(), lr=GENERAL_CFG['generator_lr_by_direction'][direction],
        weight_decay=GENERAL_CFG['weight_decay'], betas=(.5, .999),
    )
    discriminator_parameters = list(discriminator_full.parameters()) + list(discriminator_half.parameters())
    opt_d = torch.optim.AdamW(
        discriminator_parameters, lr=GENERAL_CFG['discriminator_lr'],
        weight_decay=0., betas=(.5, .999),
    )
    sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_g, T_max=target_updates, eta_min=GENERAL_CFG['generator_lr_by_direction'][direction] * .1,
    )
    sched_d = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_d, T_max=target_updates, eta_min=GENERAL_CFG['discriminator_lr'] * .1,
    )

    progress = {'step': 0, 'best_loss': float('inf'), 'best_step': 0,
                'last_loss': float('inf'), 'seconds': 0.}
    history = []
    last = folder / 'last_training_state.pt'
    if last.exists():
        state = torch.load(last, map_location='cpu', weights_only=False)
        assert state['signature'] == signature, 'General-stage configuration changed; use a new run folder.'
        generator.load_state_dict(state['generator'])
        discriminator_full.load_state_dict(state['discriminator_full'])
        discriminator_half.load_state_dict(state['discriminator_half'])
        opt_g.load_state_dict(state['opt_g']); opt_d.load_state_dict(state['opt_d'])
        sched_g.load_state_dict(state['sched_g']); sched_d.load_state_dict(state['sched_d'])
        progress, history = state['progress'], state['history']
        torch.set_rng_state(state['rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
        print('Resumed general', direction, 'at step', progress['step'])
        del state
    else:
        print('Loaded reviewed', direction, 'fallback step', initial['step'])
    del initial

    started = time.perf_counter(); elapsed_before = progress['seconds']

    def checkpoint(path, score):
        atomic_torch(path, {
            'generator': {key: value.detach().cpu() for key, value in generator.state_dict().items()},
            'approach': 'unet',
            'training': 'general_20k_multiscale_patchgan',
            'direction': direction,
            'output_mode': output_mode,
            'width': CFG['cnn_width'],
            'resolution': CFG['resolution'],
            'data_fingerprint': GENERAL_DATA_FINGERPRINT,
            'unique_coco_train_images': len(GENERAL_COCO_SPLITS['train']),
            'unique_fs2k_train_pairs': len(SPLITS['train']),
            'step': progress['step'],
            'val_lpips': score,
            'transferred_from': str(initial_checkpoint),
        })

    def save_last():
        progress['seconds'] = elapsed_before + time.perf_counter() - started
        atomic_torch(last, {
            'signature': signature,
            'generator': generator.state_dict(),
            'discriminator_full': discriminator_full.state_dict(),
            'discriminator_half': discriminator_half.state_dict(),
            'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(),
            'sched_g': sched_g.state_dict(), 'sched_d': sched_d.state_dict(),
            'progress': dict(progress), 'history': history,
            'rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
            'python_rng': random.getstate(), 'numpy_rng': np.random.get_state(),
        })
        atomic_json(folder / 'history.json', history)

    def validate():
        rows = general_validation(generator, direction)
        overall = float(np.mean([row['lpips'] for row in rows]))
        coco = float(np.mean([row['lpips'] for row in rows if row['domain'] == 'coco_synthetic']))
        fs2k = float(np.mean([row['lpips'] for row in rows if row['domain'] == 'fs2k_artist']))
        entry = {'step': progress['step'], 'val_lpips': overall,
                 'coco_lpips': coco, 'fs2k_lpips': fs2k}
        history.append(entry)
        checkpoint(folder / f"snapshot_{progress['step']:07d}.pt", overall)
        progress['last_loss'] = overall
        if overall < progress['best_loss']:
            progress.update(best_loss=overall, best_step=progress['step'])
            checkpoint(folder / 'best.pt', overall)
        atomic_json(folder / 'validation_rows_step_{:07d}.json'.format(progress['step']), rows)
        atomic_json(folder / 'selection.json', dict(progress))
        print('Validation', entry)

    if progress['step'] == 0 and not history:
        validate(); save_last()

    order = list(range(target_updates))
    # GENERAL_TRAIN_RECORDS was already deterministically shuffled. Keeping this
    # order stable makes the exact update offset reproducible after disconnection.
    bar = tqdm(total=target_updates, initial=progress['step'], desc='General 20K ' + direction)
    while progress['step'] < target_updates:
        resolution = 256 if progress['step'] < GENERAL_CFG['progressive_512_start'] else 512
        boundary = min(
            target_updates,
            GENERAL_CFG['progressive_512_start'] if resolution == 256 else target_updates,
        )
        remaining_indices = order[progress['step']:boundary]
        dataset = GeneralBidirectionalDataset(
            GENERAL_TRAIN_RECORDS, direction, resolution, training=True
        )
        loader = DataLoader(
            Subset(dataset, remaining_indices), batch_size=1, shuffle=False,
            num_workers=2, pin_memory=True, persistent_workers=True,
        )
        generator.train(); discriminator_full.train(); discriminator_half.train()
        for batch in loader:
            source = batch['source'].to(DEVICE, non_blocking=True)
            target = batch['target'].to(DEVICE, non_blocking=True)
            fake = generator(source)

            opt_d.zero_grad(set_to_none=True)
            full_real = discriminator_full(torch.cat([source, target], 1))
            full_fake = discriminator_full(torch.cat([source, fake.detach()], 1))
            half_source = F.interpolate(source, scale_factor=.5, mode='bilinear', align_corners=False)
            half_target = F.interpolate(target, scale_factor=.5, mode='bilinear', align_corners=False)
            half_fake = F.interpolate(fake.detach(), scale_factor=.5, mode='bilinear', align_corners=False)
            half_real_score = discriminator_half(torch.cat([half_source, half_target], 1))
            half_fake_score = discriminator_half(torch.cat([half_source, half_fake], 1))
            loss_d = .5 * (
                F.relu(1 - full_real).mean() + F.relu(1 + full_fake).mean()
                + F.relu(1 - half_real_score).mean() + F.relu(1 + half_fake_score).mean()
            )
            if not torch.isfinite(loss_d):
                raise FloatingPointError('General discriminator loss is not finite')
            loss_d.backward()
            nn.utils.clip_grad_norm_(discriminator_parameters, 1.)
            opt_d.step(); sched_d.step()

            for discriminator in (discriminator_full, discriminator_half):
                discriminator.eval().requires_grad_(False)
            opt_g.zero_grad(set_to_none=True)
            paired = general_direction_loss(fake, target, source, direction)
            full_adversarial = -discriminator_full(torch.cat([source, fake], 1)).mean()
            half_generated = F.interpolate(fake, scale_factor=.5, mode='bilinear', align_corners=False)
            half_adversarial = -discriminator_half(torch.cat([half_source, half_generated], 1)).mean()
            adversarial = .5 * (full_adversarial + half_adversarial)
            next_step = progress['step'] + 1
            adversarial_weight = GENERAL_CFG['adversarial_max_by_direction'][direction] * min(
                1., next_step / GENERAL_CFG['adversarial_warmup']
            )
            loss_g = paired + adversarial_weight * adversarial
            if not torch.isfinite(loss_g):
                raise FloatingPointError('General generator loss is not finite')
            loss_g.backward()
            nn.utils.clip_grad_norm_(generator.parameters(), 1., error_if_nonfinite=True)
            opt_g.step(); sched_g.step()
            for discriminator in (discriminator_full, discriminator_half):
                discriminator.train().requires_grad_(True)

            progress['step'] = next_step
            history.append({
                'step': next_step, 'resolution': resolution,
                'generator_loss': float(loss_g.detach()),
                'paired_loss': float(paired.detach()),
                'adversarial_loss': float(adversarial.detach()),
                'adversarial_weight': adversarial_weight,
                'discriminator_loss': float(loss_d.detach()),
                'lr': sched_g.get_last_lr()[0],
            })
            bar.update(1)
            bar.set_postfix(res=resolution, g=round(float(loss_g.detach()), 3),
                            d=round(float(loss_d.detach()), 3))
            if next_step % GENERAL_CFG['validate_every'] == 0 or next_step == target_updates:
                validate()
            if next_step % GENERAL_CFG['save_every'] == 0 or next_step == target_updates:
                save_last()
            if next_step == boundary:
                break
        del loader, dataset

    bar.close(); save_last()
    checkpoint(folder / 'final.pt', progress['last_loss'])
    atomic_json(folder / 'compute.json', {
        'generator_trainable_parameters': sum(parameter.numel() for parameter in generator.parameters()),
        'discriminator_trainable_parameters': sum(parameter.numel() for parameter in discriminator_parameters),
        'training_and_validation_seconds': progress['seconds'],
        'hardware': atomic_environment,
        'best_step': progress['best_step'],
        'completed_updates': progress['step'],
        'unique_coco_train_images': len(GENERAL_COCO_SPLITS['train']),
        'unique_fs2k_train_pairs': len(SPLITS['train']),
    })
    del generator, discriminator_full, discriminator_half, opt_g, opt_d, sched_g, sched_d
    gc.collect(); torch.cuda.empty_cache()
    return dict(progress)


In [ ]:
GENERAL_INITIALS = {
    direction: Path(SELECTED[direction]['unet']['path'])
    for direction in PROMPTS
}
GENERAL_PROGRESS = {}
for direction in GENERAL_CFG['direction_order']:
    GENERAL_PROGRESS[direction] = train_general_direction(
        direction, GENERAL_INITIALS[direction]
    )

GENERAL_SELECTED = {}
for direction in PROMPTS:
    folder = GENERAL_ROOT / 'unet_multiscale_patchgan' / direction
    result = GENERAL_PROGRESS[direction]
    GENERAL_SELECTED[direction] = {
        'unet': {
            'folder': str(folder),
            'path': str(folder / 'best.pt'),
            'val_lpips': result['best_loss'],
            'best_step': result['best_step'],
            'tuned': True,
            'output_mode': OUTPUT_MODES[direction],
            'training': 'general_20k_multiscale_patchgan',
            'warm_started_from': str(GENERAL_INITIALS[direction]),
        }
    }

atomic_json(GENERAL_ROOT / 'selected_models.json', GENERAL_SELECTED)
print('General bidirectional refinement complete:', json.dumps(GENERAL_SELECTED, indent=2))


## 9. General-image bidirectional visual gate

Review both directions on diverse COCO scenes **and** real FS2K portraits. For each
direction, compare the reviewed face-focused fallback, validation-selected general
model, and final general step. Photo-to-pencil must preserve composition and produce
clean line art; pencil-to-photo must produce plausible color without severe blur,
identity/composition changes, or invented objects. Do not run held-out testing until
all four figures are reviewed.


In [ ]:
def render_general_gate(direction, records, title, output_name):
    initial_path = GENERAL_INITIALS[direction]
    folder = GENERAL_ROOT / 'unet_multiscale_patchgan' / direction
    before, before_state = load_unet_checkpoint(initial_path)
    selected_path = Path(GENERAL_SELECTED[direction]['unet']['path'])
    selected, selected_state = load_unet_checkpoint(selected_path)
    final_path = folder / 'final.pt'
    final_model, final_state = load_unet_checkpoint(final_path)

    fig, axes = plt.subplots(len(records), 5, figsize=(18, 4 * len(records)), squeeze=False)
    for row_index, record in enumerate(records):
        photo, sketch = get_general_pair(record, CFG['resolution'])
        source_image, target_image = (
            (photo, sketch) if direction == 'photo_to_pencil' else (sketch, photo)
        )
        images = [
            source_image,
            target_image,
            predict_unet(before, source_image, direction),
            predict_unet(selected, source_image, direction),
            predict_unet(final_model, source_image, direction),
        ]
        labels = [
            'Input', 'Real target', 'Before general 20K',
            'LPIPS-selected step ' + str(selected_state['step']),
            'Final step ' + str(final_state['step']),
        ]
        for axis, image, label in zip(axes[row_index], images, labels):
            axis.imshow(image); axis.set_title(label); axis.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    output = GENERAL_ROOT / output_name
    fig.savefig(output, dpi=150)
    plt.show(); plt.close(fig)
    del before, selected, final_model
    gc.collect(); torch.cuda.empty_cache()
    return output


GENERAL_GATE_FIGURES = []
for direction in PROMPTS:
    GENERAL_GATE_FIGURES.append(render_general_gate(
        direction,
        GENERAL_COCO_SPLITS['val'][:4],
        direction + ' general gate - diverse COCO scenes',
        direction + '_general_coco_visual_gate.png',
    ))
    GENERAL_GATE_FIGURES.append(render_general_gate(
        direction,
        SPLITS['val'][:4],
        direction + ' general gate - held-out FS2K portraits',
        direction + '_general_fs2k_visual_gate.png',
    ))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row_index, direction in enumerate(PROMPTS):
    folder = GENERAL_ROOT / 'unet_multiscale_patchgan' / direction
    history = json.loads((folder / 'history.json').read_text())
    validation = [row for row in history if 'val_lpips' in row]
    for axis, key, title in zip(
        axes[row_index],
        ('val_lpips', 'coco_lpips', 'fs2k_lpips'),
        ('Overall LPIPS', 'COCO LPIPS', 'FS2K LPIPS'),
    ):
        axis.plot([row['step'] for row in validation], [row[key] for row in validation], marker='o')
        axis.set_title(direction + ' - ' + title)
        axis.set_xlabel('Updates'); axis.set_ylabel('Lower is better')
fig.tight_layout(); plt.show(); plt.close(fig)

print('GENERAL VISUAL GATE: inspect both directions on general scenes and faces.')
print('The earlier reviewed face checkpoints remain untouched as fallbacks.')
print('Do not run Section 10 until all four figures are reviewed.')


## 10. Shared held-out test evaluation and failure analysis

Only enable after choices are final. `TEST_LIMIT=None` evaluates the ENTIRE test set;
subsets are labeled as subsets. All three selected trained approaches use the same
test IDs, 512px preprocessing and seed schedule. Results are per direction AND data
domain, plus a sample-weighted overall row. Check domain scores: COCO majority can
hide poor performance on real pencil drawings. No restoration path or saturation
adjustment is applied. Baseline pretrained-only performance is not claimed as training.

LPIPS, MAE, SSIM and PSNR quantify paired similarity, not absolute artistic quality.
Colourfulness is descriptive only; oversaturation is not success. A photo with
plausible different clothing colours may receive lower pixel similarity. Test identity
fidelity and generalization with independent drawings too; document rights and origin.


In [ ]:
RUN_TEST_EVALUATION = False
TEST_LIMIT = None  # full held-out test; partial runs are labeled and are not the final result
TEST_REVIEW_NOTE = ''

if RUN_TEST_EVALUATION:
    assert FINAL_MODELS_APPROVED, 'Visual validation has not approved final models.'
    test_approval = input(
        'Inspect the validation curves/comparisons above. If model choices are final, type TEST to unlock the held-out test: '
    ).strip()
    if test_approval != 'TEST':
        raise RuntimeError('Held-out test remains locked. Training checkpoints are safe; review validation and resume here.')
    TEST_REVIEW_NOTE = 'Validation results reviewed; model choices frozen before held-out test.'

def colorfulness(image):
    a=np.asarray(image,dtype=np.float32); rg=a[...,0]-a[...,1]; yb=.5*(a[...,0]+a[...,1])-a[...,2]
    return float(np.sqrt(rg.var()+yb.var())+.3*np.sqrt(rg.mean()**2+yb.mean()**2))

if RUN_TEST_EVALUATION:
    assert FINAL_MODELS_APPROVED, 'Visual validation has not approved final models.'
    assert TEST_REVIEW_NOTE.strip(), 'Record your validation review before unlocking the test set.'
    from skimage.metrics import structural_similarity, peak_signal_noise_ratio
    records = SPLITS['test'] if TEST_LIMIT is None else SPLITS['test'][:TEST_LIMIT]
    assert records
    frozen = {'selected':SELECTED,'test_ids':[r['id'] for r in records],'data':DATA_FINGERPRINT,
              'inference_steps':CFG['inference_steps'],'seed':SEED,'implementation':IMPLEMENTATION_SHA}
    frozen_path=REPORTS/'frozen_test_protocol.json'
    if frozen_path.exists(): assert json.loads(frozen_path.read_text())==frozen, 'Test protocol changed; do not mix results.'
    else: atomic_json(frozen_path,frozen)
    aggregate=[]
    for direction in PROMPTS:
        for approach in ('unet','gan','diffusion'):
            result_file=REPORTS/(direction+'_'+approach+'_test.json')
            rows=json.loads(result_file.read_text()) if result_file.exists() else []
            assert [r['id'] for r in rows]==[r['id'] for r in records[:len(rows)]]
            model=load_selected(direction,approach)
            for i in tqdm(range(len(rows),len(records)),desc=direction+' '+approach):
                record=records[i]; photo,sketch,box=get_pair(record)
                source,target=(photo,sketch) if direction=='photo_to_pencil' else (sketch,photo)
                pred=predict_canvas(model,approach,source,direction,i).crop(box); target=target.crop(box)
                p,y=np.asarray(pred),np.asarray(target)
                mse=float(np.mean((p.astype(np.float64)-y.astype(np.float64))**2))
                # Cap exact-match PSNR at 100 dB for finite CSV/JSON, document the cap.
                rows.append({'id':record['id'],'domain':record['domain'],'lpips':perceptual_score(pred,target),
                    'mae':float(np.abs(p.astype(float)-y).mean()/255),'psnr':min(100.,10*math.log10(255**2/max(mse,1e-10))),
                    'ssim':float(structural_similarity(y,p,channel_axis=2,data_range=255)),
                    'pred_colorfulness':colorfulness(pred),'target_colorfulness':colorfulness(target)})
                if (i+1)%20==0: atomic_json(result_file,rows)
            atomic_json(result_file,rows)
            compute=json.loads((Path(SELECTED[direction][approach]['folder'])/'compute.json').read_text())
            for domain in ['overall']+sorted({r['domain'] for r in rows}):
                group=rows if domain=='overall' else [r for r in rows if r['domain']==domain]
                aggregate.append({'direction':direction,'approach':approach,'domain':domain,'count':len(group),
                    'full_test':len(records)==len(SPLITS['test']),'selected_step':SELECTED[direction][approach]['best_step'],
                    'generator_parameters':compute['generator_trainable_parameters'],
                    'discriminator_parameters':compute['discriminator_trainable_parameters'],
                    'training_hours':compute['training_and_validation_seconds']/3600,
                    **{k:float(np.mean([r[k] for r in group])) for k in ('lpips','mae','psnr','ssim','pred_colorfulness','target_colorfulness')}})
            worst=sorted(range(len(rows)),key=lambda i:rows[i]['lpips'],reverse=True)[:6]
            fig,axes=plt.subplots(len(worst),3,figsize=(10,3*len(worst)),squeeze=False)
            for row_index,i in enumerate(worst):
                r=records[i]; photo,sketch,box=get_pair(r); source,target=(photo,sketch) if direction=='photo_to_pencil' else (sketch,photo)
                pred=predict_canvas(model,approach,source,direction,i)
                for ax,im,label in zip(axes[row_index],[source,pred,target],['Input','Worst LPIPS '+str(round(rows[i]['lpips'],3)),'Target']):
                    ax.imshow(im.crop(box)); ax.set_title(label); ax.axis('off')
            fig.tight_layout(); fig.savefig(REPORTS/(direction+'_'+approach+'_failure_cases.png'),dpi=140); plt.close(fig)
            atomic_json(REPORTS/(direction+'_'+approach+'_failure_ids.json'),[rows[i] for i in worst])
            del model; gc.collect(); torch.cuda.empty_cache()
    atomic_json(REPORTS/'comparison_metrics.json',aggregate)
    with (REPORTS/'comparison_metrics.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(aggregate[0])); writer.writeheader(); writer.writerows(aggregate)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,direction in zip(axes,PROMPTS):
        rows=[r for r in aggregate if r['direction']==direction and r['domain']=='overall']
        ax.bar([r['approach'] for r in rows],[r['lpips'] for r in rows]); ax.set_title(direction); ax.set_ylabel('Test LPIPS (lower better)')
    fig.tight_layout(); fig.savefig(REPORTS/'test_metric_comparison.png',dpi=150); plt.show(); plt.close(fig)
    print('Saved measured results, comparison plots and worst cases. Training hours exclude hyperparameter trials; report their CSV separately.')
else:
    print('Test not run. No test accuracy or improvement claim is available.')


## 11. Export all selected approaches; one deployable Streamlit app

Exports include all six selected models, the validation-selected defaults, paired
preprocessing and exact source. Old `.pt` files are untouched. A pilot export is
clearly marked pilot. Step-zero selections are also recorded: pretrained or random
initialization winning is a failure to improve, not a successful trained result.
Do NOT replace your old deployment with a failed or visually unacceptable run.

Extract the ZIP CONTENTS to `checkpoints/quality_study/` in your project. Install
`requirements-diffusion.txt` in a Python 3.11/3.12 environment, then run:

```bash
python app.py
```

Use the Streamlit sidebar to choose the new or original model source. Plain
`python app.py` uses the new default bundle if present, otherwise the old checkpoints.
There is only one app launcher. The export also contains a portable copy and the
required source modules, so it can run without your existing project. The portable
export shows only the new models. The pretrained diffusion base downloads on first
use; adapters alone are not full models.


In [ ]:
assert FINAL_MODELS_APPROVED, 'Do not export failed or visually unapproved models.'
assert all(DEPLOY_APPROACH.get(direction) in ('unet','gan','diffusion') for direction in PROMPTS), \
    'Choose one visually approved DEPLOY_APPROACH for each direction first.'
EXPORT=STUDY_ROOT/'quality_study'; EXPORT.mkdir(exist_ok=True)
models={}; defaults={}
for direction in PROMPTS:
    models[direction]={}; defaults[direction]=DEPLOY_APPROACH[direction]
    for approach,info in SELECTED[direction].items():
        source=Path(info['path']); destination=EXPORT/'models'/direction/approach
        destination.mkdir(parents=True,exist_ok=True)
        if approach=='diffusion':
            for name in ('adapter_config.json','adapter_model.safetensors','selection.json'): shutil.copy2(source/name,destination/name)
            relative=str(destination.relative_to(EXPORT))
        else:
            shutil.copy2(source,destination/'best.pt'); relative=str((destination/'best.pt').relative_to(EXPORT))
        models[direction][approach]={'path':relative,'val_lpips':info['val_lpips'],'best_step':info['best_step'],'tuned':info['tuned']}
        report_dir=EXPORT/'reports'/direction/approach; report_dir.mkdir(parents=True,exist_ok=True)
        for name in ('history.json','recipe.json','compute.json','selection.json'):
            path=Path(info['folder'])/name
            if path.exists(): shutil.copy2(path,report_dir/name)
manifest={'format':'sketch2photo-study-v2','mode':MODE,'models':models,'validation_best':defaults,
    'base_model':BASE_MODEL,'base_revision':BASE_REVISION,'data_fingerprint':DATA_FINGERPRINT,
    'data_source':DATA_SOURCE,'counts':dict(Counter(r['split'] for r in RECORDS)),
    'inference':{'size':CFG['resolution'],'steps':CFG['inference_steps'],'text_guidance':6.5,'image_guidance':2.2},
    'quality_status':'Unproven until measured and visually reviewed; no perfect-output guarantee'}
atomic_json(EXPORT/'manifest.json',manifest)
for relative,content in MODULE_SOURCES.items():
    target=EXPORT/relative; target.parent.mkdir(parents=True,exist_ok=True); target.write_text(content)

(EXPORT/"app.py").write_text('"""Streamlit web application for original checkpoints and new Colab exports."""\nfrom __future__ import annotations\n\nimport io\nimport os\nimport sys\nfrom pathlib import Path\n\nfrom PIL import Image\nimport streamlit as st\n\n\nROOT = Path(__file__).resolve().parent\nDEFAULT_BUNDLE = Path(os.getenv(\'SKETCH2PHOTO_BUNDLE\', ROOT / \'checkpoints\' / \'quality_study\'))\nif (ROOT / \'manifest.json\').is_file():\n    DEFAULT_BUNDLE = ROOT\n\n\ndef png_bytes(image: Image.Image) -> bytes:\n    buffer = io.BytesIO()\n    image.save(buffer, format=\'PNG\')\n    return buffer.getvalue()\n\n\ndef uploaded_image(uploaded) -> Image.Image | None:\n    if uploaded is None:\n        return None\n    with Image.open(uploaded) as image:\n        return image.copy()\n\n\ndef render_original(st):\n    """Original 128px checkpoints plus the classical pencil renderer."""\n    from src import application as legacy\n\n    st.info(\n        \'Original mode uses the supplied 128×128 sketch-to-photo checkpoints. \'\n        \'Photo-to-pencil is the existing image-processing renderer, not a trained model.\'\n    )\n    checkpoints = legacy.list_checkpoints()\n    with st.sidebar:\n        st.subheader(\'Original model\')\n        checkpoint = st.selectbox(\n            \'Checkpoint\', checkpoints,\n            index=checkpoints.index(legacy.default_checkpoint(checkpoints)) if checkpoints else None,\n            disabled=not checkpoints,\n        )\n        guessed = legacy.guess_arch(checkpoint)\n        architecture = st.selectbox(\n            \'Architecture\', legacy.ARCH_NAMES,\n            index=legacy.ARCH_NAMES.index(guessed),\n        )\n        device = st.selectbox(\'Compute device\', legacy.AVAILABLE_DEVICES,\n                              index=legacy.AVAILABLE_DEVICES.index(legacy.DEFAULT_DEVICE))\n\n    pencil_tab, colour_tab = st.tabs([\'Photo → Pencil\', \'Pencil → Colour Photo\'])\n    with pencil_tab:\n        st.subheader(\'Create a pencil drawing\')\n        st.caption(\'The preview and downloaded PNG are exactly the same image.\')\n        source_file = st.file_uploader(\'Upload a photo\', type=[\'png\', \'jpg\', \'jpeg\', \'webp\'], key=\'old_photo\')\n        c1, c2, c3 = st.columns(3)\n        darkness = c1.slider(\'Pencil darkness\', .5, 2.0, 1.05, .05)\n        fade = c2.slider(\'Fade soft scenery\', 0.0, 1.0, .6, .05)\n        texture = c3.slider(\'Paper and pencil texture\', 0.0, 1.0, .45, .05)\n        if st.button(\'Create pencil drawing\', type=\'primary\', use_container_width=True):\n            image = uploaded_image(source_file)\n            if image is None:\n                st.error(\'Upload a photo first.\')\n            else:\n                with st.spinner(\'Rendering pencil drawing…\'):\n                    drawing, technical, pair = legacy.prepare_drawing_session(image, darkness, fade, texture)\n                st.session_state.old_drawing = drawing\n                st.session_state.old_technical = technical\n                st.session_state.old_pair = pair\n        drawing = st.session_state.get(\'old_drawing\')\n        if drawing is not None:\n            left, right = st.columns(2)\n            source = st.session_state[\'old_pair\'][\'source\']\n            left.image(source, caption=\'Source photo\', use_container_width=True)\n            right.image(drawing, caption=\'Pencil drawing\', use_container_width=True)\n            st.download_button(\'Download pencil PNG\', png_bytes(drawing), \'pencil_drawing.png\',\n                               \'image/png\', use_container_width=True)\n            with st.expander(\'Technical input used by the original checkpoint\'):\n                st.image(st.session_state[\'old_technical\'], width=320)\n\n    with colour_tab:\n        st.subheader(\'Generate a colour photo\')\n        use_created = st.checkbox(\'Use the pencil drawing created above\',\n                                  value=st.session_state.get(\'old_drawing\') is not None,\n                                  disabled=st.session_state.get(\'old_drawing\') is None)\n        colour_file = None if use_created else st.file_uploader(\n            \'Upload a white-paper pencil sketch or a photo\', type=[\'png\', \'jpg\', \'jpeg\', \'webp\'], key=\'old_colour\')\n        input_kind = st.radio(\'Input type\', [legacy.INPUT_PENCIL, legacy.INPUT_PHOTO], horizontal=True,\n                              disabled=use_created)\n        raw_colours = st.toggle(\'Raw model colours (recommended for evaluation)\', value=True)\n        if raw_colours:\n            colour_strength, cast_reduction = 1.0, 0.0\n        else:\n            c1, c2 = st.columns(2)\n            colour_strength = c1.slider(\'Display colour intensity\', 0.0, 2.5, 1.5, .05)\n            cast_reduction = c2.slider(\'Reduce cream/yellow cast\', 0.0, 1.0, .35, .05)\n        if st.button(\'Generate colour photo\', type=\'primary\', use_container_width=True):\n            image = st.session_state.get(\'old_drawing\') if use_created else uploaded_image(colour_file)\n            if image is None:\n                st.error(\'Create or upload a pencil drawing first.\')\n            elif not checkpoint:\n                st.error(\'No original checkpoint is available.\')\n            else:\n                try:\n                    with st.spinner(\'Running the selected checkpoint…\'):\n                        result, technical, details = legacy.run_inference(\n                            image, legacy.INPUT_PENCIL if use_created else input_kind,\n                            legacy.OUTPUT_MODEL, checkpoint, architecture, device,\n                            st.session_state.get(\'old_pair\') if use_created else None,\n                            colour_strength, cast_reduction,\n                        )\n                    st.session_state.old_colour_result = result\n                    st.session_state.old_colour_input = technical\n                    st.session_state.old_colour_details = details\n                except Exception as error:\n                    st.error(str(error))\n        result = st.session_state.get(\'old_colour_result\')\n        if result is not None:\n            left, right = st.columns(2)\n            input_image = st.session_state.get(\'old_drawing\') if use_created else uploaded_image(colour_file)\n            if input_image is not None:\n                left.image(input_image, caption=\'Pencil input\', use_container_width=True)\n            right.image(result, caption=\'Generated colour result\', use_container_width=True)\n            st.download_button(\'Download colour PNG\', png_bytes(result), \'colour_result.png\',\n                               \'image/png\', use_container_width=True)\n            st.caption(st.session_state.get(\'old_colour_details\', \'\'))\n            with st.expander(\'Technical network input\'):\n                st.image(st.session_state.get(\'old_colour_input\'), width=320)\n\n\n@st.cache_resource(show_spinner=False)\ndef get_study_editor(bundle: str):\n    from src.study_backend import StudyEditor\n    return StudyEditor(bundle)\n\n\n@st.cache_resource(show_spinner=False)\ndef get_dual_editor(bundle: str):\n    from src.diffusion_backend import DualEditor\n    return DualEditor(bundle)\n\n\ndef render_new(st, bundle: Path):\n    """New three-approach study bundle or earlier two-adapter export."""\n    import json\n    manifest_path = bundle / \'manifest.json\'\n    if not manifest_path.is_file():\n        st.warning(\n            \'No new trained export is installed yet. Run the master Colab notebook and \'\n            f\'extract the ZIP contents into `{bundle}`.\'\n        )\n        return\n    manifest = json.loads(manifest_path.read_text())\n    model_format = manifest.get(\'format\')\n    if model_format == \'sketch2photo-study-v2\':\n        editor = get_study_editor(str(bundle))\n        approaches = [\'Validation best\', \'U-Net\', \'GAN\', \'Diffusion\']\n    elif model_format == \'sketch2photo-ip2p-lora-v1\':\n        editor = get_dual_editor(str(bundle))\n        approaches = [\'Diffusion\']\n    else:\n        st.error(\'The selected folder is not a recognized Sketch2Photo export.\')\n        return\n\n    mode = manifest.get(\'mode\', \'unknown\')\n    st.info(f\'Loaded a **{mode}** export. “Validation best” means lowest held-out validation LPIPS, not perfect output.\')\n    direction_label = st.segmented_control(\n        \'Direction\', [\'Photo → Pencil\', \'Pencil → Colour Photo\'], default=\'Photo → Pencil\')\n    direction = \'photo_to_pencil\' if direction_label == \'Photo → Pencil\' else \'sketch_to_photo\'\n    uploaded = st.file_uploader(\'Upload source image\', type=[\'png\', \'jpg\', \'jpeg\', \'webp\'], key=\'new_source\')\n    c1, c2 = st.columns(2)\n    approach_label = c1.selectbox(\'Trained approach\', approaches)\n    seed = c2.number_input(\'Diffusion seed\', min_value=0, max_value=2**31-1, value=42, step=1)\n    description = st.text_input(\'Optional colour/content guidance (diffusion only)\',\n                                placeholder=\'Example: red jacket, green plants, warm daylight\')\n    if st.button(\'Generate result\', type=\'primary\', use_container_width=True):\n        image = uploaded_image(uploaded)\n        if image is None:\n            st.error(\'Upload an image first.\')\n        else:\n            mapping = {\'Validation best\':\'validation_best\', \'U-Net\':\'unet\', \'GAN\':\'gan\', \'Diffusion\':\'diffusion\'}\n            try:\n                with st.spinner(\'Generating with the trained model…\'):\n                    if model_format == \'sketch2photo-study-v2\':\n                        result = editor(image, direction, mapping[approach_label], int(seed), description)\n                    else:\n                        result = editor(image, direction, int(seed), description)\n                st.session_state.new_result = result\n                st.session_state.new_input = image\n                st.session_state.new_direction = direction\n            except Exception as error:\n                st.error(str(error))\n    result = st.session_state.get(\'new_result\')\n    if result is not None:\n        left, right = st.columns(2)\n        left.image(st.session_state[\'new_input\'], caption=\'Input\', use_container_width=True)\n        right.image(result, caption=\'Actual generated output\', use_container_width=True)\n        output_direction = st.session_state.get(\'new_direction\', direction)\n        filename = \'pencil_drawing.png\' if output_direction == \'photo_to_pencil\' else \'colour_photo.png\'\n        st.download_button(\'Download generated PNG\', png_bytes(result), filename, \'image/png\',\n                           use_container_width=True)\n        st.caption(\'The displayed preview and downloaded file are the same model output.\')\n\n\ndef render():\n    st.set_page_config(page_title=\'Sketch2Photo Studio\', page_icon=\'✏️\', layout=\'wide\')\n    st.markdown(\'\'\'<style>\n    .block-container {max-width: 1240px; padding-top: 2rem;}\n    [data-testid="stHeader"] {background: rgba(0,0,0,0);}\n    .hero {padding: 1.25rem 1.4rem; border-radius: 18px; margin-bottom: 1rem;\n      background: linear-gradient(120deg,#201b2c,#522b47 55%,#8a4a2d); color:white;}\n    .hero h1 {margin:0; font-size:2.25rem;} .hero p {margin:.35rem 0 0; opacity:.85;}\n    </style><div class="hero"><h1>Sketch2Photo Studio</h1><p>Train once in Colab. Compare models and create pencil drawings or colour photographs on the web.</p></div>\'\'\',\n                unsafe_allow_html=True)\n    bundle_text = st.sidebar.text_input(\'New model bundle\', str(DEFAULT_BUNDLE))\n    bundle = Path(bundle_text).expanduser().resolve()\n    new_available = (bundle / \'manifest.json\').is_file()\n    legacy_available = (ROOT / \'src\' / \'application.py\').is_file() and (ROOT / \'src\' / \'models.py\').is_file()\n    choices = [\'New trained models\'] + ([\'Original checkpoints\'] if legacy_available else [])\n    engine = st.sidebar.radio(\'Model source\', choices, index=0 if new_available or not legacy_available else 1)\n    st.sidebar.caption(\'The original weights are kept. Install a Colab export to enable the new trained models.\')\n    st.sidebar.divider()\n    st.sidebar.caption(\'PyTorch · Streamlit · 512px new workflow\')\n    if engine == \'New trained models\':\n        render_new(st, bundle)\n    else:\n        render_original(st)\n    st.divider()\n    st.caption(\'A grayscale sketch does not uniquely contain the original colours. Results are model estimates.\')\n\n\nif __name__ == \'__main__\':\n    try:\n        from streamlit.runtime.scriptrunner import get_script_run_ctx\n        running_in_streamlit = get_script_run_ctx(suppress_warning=True) is not None\n    except ImportError:\n        running_in_streamlit = False\n    if running_in_streamlit:\n        render()\n    else:\n        os.execv(sys.executable, [sys.executable, \'-m\', \'streamlit\', \'run\', str(Path(__file__).resolve()), *sys.argv[1:]])\n')
(EXPORT/"requirements.txt").write_text('# Reproducible local inference environment; use Python 3.11 or 3.12.\ntorch==2.8.0\ndiffusers==0.35.1\ntransformers==4.55.4\npeft==0.17.1\naccelerate==1.10.1\nhuggingface-hub==0.34.4\nsafetensors==0.6.2\nPillow==11.3.0\nstreamlit==1.65.0\n')

for name in ('environment.json','requirements-lock.txt','tuning_trials.json','tuning_trials.csv','validation_selection_ids.json'):
    path=STUDY_ROOT/name
    if path.exists(): shutil.copy2(path,EXPORT/'reports'/name)
for path in REPORTS.iterdir():
    if path.is_file(): shutil.copy2(path,EXPORT/'reports'/path.name)
shutil.copy2(STUDY_ROOT/'training_pairs.png',EXPORT/'reports'/'training_pairs.png')
archive=shutil.make_archive(str(STUDY_ROOT/('quality_study_'+MODE)),'zip',EXPORT)
print('Model export:',archive)
print('Resume files and full data manifest remain at:',STUDY_ROOT)
print('In the exported folder run: python app.py')
if any(info['best_step']==0 for dirs in models.values() for info in dirs.values()):
    print('WARNING: at least one selected model stayed at initialization. Inspect results; do not claim it improved.')
from google.colab import files
files.download(archive)


## 12. Course submission checklist — complete after actual training

- [ ] Verify student name, approved topic and the expanded compute scope with the teacher.
- [ ] Report exact datasets, terms, split counts, style/domain distributions and biases.
- [ ] Include all three approach definitions, training recipes and saved environment lock.
- [ ] Include training/validation curves and the LR/weight-decay tuning table.
- [ ] Include the shared full-test metrics table, comparison figures and failure examples.
- [ ] Explain why the measured winner differs using skip connections, adversarial training,
  perceptual losses and transfer learning. Do not assume diffusion wins.
- [ ] Discuss uncertainty of colours, identity changes, limited artist-pair coverage,
  synthetic-target bias and possible pretrained-data overlap.
- [ ] Provide working external download links for weights larger than 50 MB, not fake URLs.
- [ ] Add the measured results and reproducible train/run commands to your README.
- [ ] Prepare 10–20 slides and the required presentation using actual measured results.
- [ ] Make genuine development commits; no fabricated history. Disclose AI assistance.

The notebook implements an experiment; it does not complete unrun results, slides,
public hosting or teacher approval. Those remain your submission steps after training.

## Technical sources and disclosure

- [U-Net paper](https://arxiv.org/abs/1505.04597)
- [pix2pix/CycleGAN official project](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix)
- [LPIPS official implementation](https://github.com/richzhang/PerceptualSimilarity)
- [InstructPix2Pix project](https://www.timothybrooks.com/instruct-pix2pix/)
- [Pretrained editor weights](https://huggingface.co/timbrooks/instruct-pix2pix)
- [Diffusers training reference](https://github.com/huggingface/diffusers/blob/v0.35.1/examples/instruct_pix2pix/train_instruct_pix2pix.py)
- [COCO official dataset and terms](https://cocodataset.org/#download)
- [FS2K official dataset and paper citation](https://github.com/DengPingFan/FS2K)

Code and experiment design are AI-assisted; the student must understand, run, verify
and report the work. No pretrained result is presented as a model trained by the student.
